In [1]:
!pip install ultralytics > /dev/null || pip uninstall -y albumentations > /dev/null || pip install albumentationsx > /dev/null || echo -e "INSTALLATION COMPLETED"

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
!pwd

CUDA available: True
CUDA device: Tesla P100-PCIE-16GB
/kaggle/working


In [3]:
%%writefile data.yaml
train:
- /kaggle/input/aod-4-yolo/train/images
val:
- /kaggle/input/aod-4-yolo/valid/images
test:
- /kaggle/input/aod-4-yolo/test/images

names:
    0: airplane
    1: bird
    2: drone
    3: helicopter

Writing data.yaml


In [4]:
%%writefile custom_yolov11n.yaml
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# Ultralytics YOLO11 object detection model with P3/8 - P5/32 outputs
# Model docs: https://docs.ultralytics.com/models/yolo11
# Task docs: https://docs.ultralytics.com/tasks/detect

# Parameters

nc: 4 # [airplane, bird, drone, helicopter]
# CHANGES: nc only

scales: # model compound scaling constants, i.e. 'model=yolo11n.yaml' will call yolo11.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.50, 0.25, 1024] # summary: 181 layers, 2624080 parameters, 2624064 gradients, 6.6 GFLOPs
  s: [0.50, 0.50, 1024] # summary: 181 layers, 9458752 parameters, 9458736 gradients, 21.7 GFLOPs
  m: [0.50, 1.00, 512] # summary: 231 layers, 20114688 parameters, 20114672 gradients, 68.5 GFLOPs
  l: [1.00, 1.00, 512] # summary: 357 layers, 25372160 parameters, 25372144 gradients, 87.6 GFLOPs
  x: [1.00, 1.50, 512] # summary: 357 layers, 56966176 parameters, 56966160 gradients, 196.0 GFLOPs

# YOLO11n backbone
backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]] # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]] # 5-P4/16
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]] # 7-P5/32
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]] # 9
  - [-1, 2, C2PSA, [1024]] # 10

# YOLO11n head
head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] # cat backbone P4
  - [-1, 2, C3k2, [512, False]] # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] # cat backbone P3
  - [-1, 2, C3k2, [256, False]] # 16 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]] # cat head P4
  - [-1, 2, C3k2, [512, False]] # 19 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]] # cat head P5
  - [-1, 2, C3k2, [1024, True]] # 22 (P5/32-large)

  - [[16, 19, 22], 1, Detect, [nc]] # Detect(P3, P4, P5)

Writing custom_yolov11n.yaml


In [5]:
# train_loop.py
import argparse
import glob
import os
from dataclasses import dataclass
from types import SimpleNamespace

import pandas as pd
import torch
from torch import Tensor
from torch.nn.utils.rnn import pad_sequence
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
from torchvision.io import decode_image
from tqdm.auto import tqdm

from ultralytics import YOLO
from ultralytics.utils.loss import v8DetectionLoss


# -----------------------------
# Utils
# -----------------------------
def get_device() -> torch.device:
    # Force cuda:0 on Kaggle when available
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


# -----------------------------
# Dataset
# -----------------------------
class AodDataset(Dataset):
    """
    Read images + YOLO labels (cls xc yc w h) normalized [0..1].

    Return:
      - image: uint8 tensor [C,H,W] on CPU
      - targets: float tensor [N,5] on CPU (or [[-1]*5] if empty/missing).
    """

    def __init__(
        self,
        images_dir: str,
        labels_dir: str,
        images_ext: str = "jpg",
        labels_ext: str = "txt",
        transforms=None,
        verbose_bad_labels: bool = False,
    ):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.images_ext = images_ext
        self.labels_ext = labels_ext
        self.transforms = transforms
        self.verbose_bad_labels = verbose_bad_labels

        img_glob = os.path.join(images_dir, f"*.{images_ext}")
        self.images = sorted(glob.glob(img_glob))
        if len(self.images) == 0:
            raise FileNotFoundError(f"No images found: {img_glob}")

        self.miss_labeled: list[str] = []

    def _labels_path_from_image(self, image_path: str) -> str:
        base = os.path.basename(image_path)
        stem = base[: -(len(self.images_ext) + 1)]
        return os.path.join(self.labels_dir, f"{stem}.{self.labels_ext}")

    def _read_labels(self, labels_path: str) -> Tensor:
        if not os.path.exists(labels_path):
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        labels: list[list[float]] = []
        with open(labels_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) != 5:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid values count: {line}")
                    continue
                try:
                    c, xc, yc, w, h = map(float, parts)
                except ValueError:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: cannot parse: {line}")
                    continue

                if not (0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid bbox: {line}")
                    continue

                labels.append([c, xc, yc, w, h])

        if len(labels) == 0:
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        return torch.tensor(labels, dtype=torch.float32)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        image_path = self.images[idx]
        img = decode_image(image_path)  # uint8 [C,H,W] on CPU
        if self.transforms is not None:
            img = self.transforms(img)

        labels_path = self._labels_path_from_image(image_path)
        targets = self._read_labels(labels_path)
        return img, targets

    def __len__(self) -> int:
        return len(self.images)


# -----------------------------
# Collate & Batch builder
# -----------------------------
def collate_fn(batch: list[tuple[Tensor, Tensor]]) -> tuple[Tensor, Tensor]:
    images, targets = zip(*batch)
    images = torch.stack(images, dim=0)
    padded_targets = pad_sequence(list(targets), batch_first=True, padding_value=-1.0)
    return images, padded_targets


def prepare_batch_dict(images: Tensor, targets: Tensor) -> dict:
    valid_mask = targets[..., 0] != -1
    if not valid_mask.any():
        return {
            "img": images,
            "batch_idx": torch.empty((0,), device=images.device, dtype=torch.int64),
            "cls": torch.empty((0,), device=images.device, dtype=torch.float32),
            "bboxes": torch.empty((0, 4), device=images.device, dtype=torch.float32),
        }

    valid_targets = targets[valid_mask]  # [M,5]

    per_img_counts = valid_mask.sum(dim=1).to(torch.int64)
    batch_idx_list = []
    for i, cnt in enumerate(per_img_counts.tolist()):
        if cnt > 0:
            batch_idx_list.append(torch.full((cnt,), i, device=images.device, dtype=torch.int64))
    batch_idx = (
        torch.cat(batch_idx_list, dim=0)
        if batch_idx_list
        else torch.empty((0,), device=images.device, dtype=torch.int64)
    )

    cls = valid_targets[:, 0].to(torch.float32)
    bboxes = valid_targets[:, 1:].to(torch.float32)
    return {"img": images, "batch_idx": batch_idx, "cls": cls, "bboxes": bboxes}


# -----------------------------
# Train / Valid
# -----------------------------
def train_one_epoch(
    model: YOLO,
    optimizer: Adam,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_train_steps: int = 0,
    debug_print_first_batch: bool = True,
) -> pd.DataFrame:
    model.model.train(True)
    torch.set_grad_enabled(True)
    loss_fn = v8DetectionLoss(model.model)

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"train epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_train_steps > 0 and step >= max_train_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        if debug_print_first_batch and step == 0:
            print("[DEBUG] images device:", images.device)
            print("[DEBUG] targets device:", targets.device)
            print("[DEBUG] model device:", next(model.model.parameters()).device)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            pbar.set_postfix_str("skip(empty)")
            continue

        optimizer.zero_grad(set_to_none=True)

        #  Prevent inference_mode leakage from Ultralytics internals in notebooks
        with torch.inference_mode(False):
            with torch.enable_grad():
                preds = model.model(images)
                batch_loss, last_loss = loss_fn(preds, batch)
                loss_scalar = batch_loss.sum()

        if not torch.isfinite(loss_scalar):
            pbar.set_postfix_str("skip(non-finite)")
            continue

        if not loss_scalar.requires_grad:
            raise RuntimeError(
                f"loss_scalar.requires_grad=False at step {step}. "
                f"Grad is disabled globally or inference_mode is active."
            )

        loss_scalar.backward()
        optimizer.step()

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        row = {
            "epoch": epoch,
            "step": step,
            "box": float(box_loss),
            "cls": float(cls_loss),
            "dfl": float(dfl_loss),
        }
        rows.append(row)
        pbar.set_postfix({"box": f"{row['box']:.4f}", "cls": f"{row['cls']:.4f}", "dfl": f"{row['dfl']:.4f}"})

    return pd.DataFrame(rows)


@torch.no_grad()
def valid_loss_only(
    model: YOLO,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_val_steps: int = 0,
) -> pd.DataFrame:
    model.model.eval()
    loss_fn = v8DetectionLoss(model.model)

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"valid(loss) epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_val_steps > 0 and step >= max_val_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            continue

        preds = model.model(images)
        _, last_loss = loss_fn(preds, batch)

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        rows.append(
            {"epoch": epoch, "batch": step, "box": float(box_loss), "cls": float(cls_loss), "dfl": float(dfl_loss)}
        )

    return pd.DataFrame(rows)


def yolo_val_metrics(model: YOLO, data_yaml: str, device_id=0, workers: int = 0):
    # plots=False để tránh warning matplotlib & nhẹ hơn
    return model.val(data=data_yaml, device=device_id, workers=workers, verbose=False, plots=False)


# -----------------------------
# Main
# -----------------------------
@dataclass
class Config:
    base_path: str
    data_yaml: str
    epochs: int = 2
    batch_size: int = 4
    lr: float = 3e-4
    num_workers: int = 2
    img_ext: str = "jpg"
    label_ext: str = "txt"
    checkpoint_dir: str = "checkpoints"

    # Debug helpers (0 means "no limit")
    max_train_steps: int = 0
    max_val_steps: int = 0


def main(cfg: Config) -> None:
    device = get_device()
    print(f"[INFO] device = {device}")
    if device.type == "cuda":
        print("[INFO] CUDA available:", torch.cuda.is_available())
        print("[INFO] CUDA device:", torch.cuda.get_device_name(0))

    ensure_dir(cfg.checkpoint_dir)

    model = YOLO(model="custom_yolov11n.yaml", task="detect", verbose=True)
    model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore
    model.model.to(device)  # type: ignore

    print("[DEBUG] model param device:", next(model.model.parameters()).device)

    optimizer = Adam(model.model.parameters(), lr=cfg.lr)

    train_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "train/images"),
        labels_dir=os.path.join(cfg.base_path, "train/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )
    val_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "valid/images"),
        labels_dir=os.path.join(cfg.base_path, "valid/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )

    pin = torch.cuda.is_available()
    workers = cfg.num_workers

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )

    train_all = []

    best_score = float("inf")
    best_epoch = -1
    best_path = os.path.join(cfg.checkpoint_dir, "best.pt")
    last_path = os.path.join(cfg.checkpoint_dir, "last.pt")

    for epoch in range(1, cfg.epochs + 1):
        print(f"\n==================== EPOCH {epoch}/{cfg.epochs} ====================")

        train_df = train_one_epoch(
            model,
            optimizer,
            train_loader,
            device,
            epoch,
            max_train_steps=cfg.max_train_steps,
            debug_print_first_batch=(epoch == 1),
        )
        train_all.append(train_df)

        if len(train_df) > 0:
            train_score = train_df[["box", "cls", "dfl"]].mean().sum()
            print(f"[EPOCH {epoch}] train_score={train_score:.6f} | best_score={best_score:.6f}")

            if train_score < best_score:
                best_score = train_score
                best_epoch = epoch
                model.save(best_path)
                print(f"[BEST] updated → epoch {best_epoch}, saved to {best_path}")
        else:
            print("[TRAIN] no valid batches produced loss (all empty labels?)")

        model.save(last_path)
        ckpt_path = os.path.join(cfg.checkpoint_dir, f"epoch_{epoch}.pt")
        model.save(ckpt_path)
        print(f"[INFO] saved checkpoint: {ckpt_path} (and last.pt)")

    train_csv = os.path.join(cfg.checkpoint_dir, "train_losses.csv")
    pd.concat(train_all, axis=0, ignore_index=True).to_csv(train_csv, index=False)
    print(f"[INFO] saved train logs: {train_csv}")

    print("\n==================== TRAIN SUMMARY ====================")
    print(f"[BEST MODEL] epoch = {best_epoch}")
    print(f"[BEST MODEL] score = {best_score:.6f}")
    print(f"[BEST MODEL] path  = {best_path}")
    print(f"[LAST MODEL] path  = {last_path}")

    print("\n==================== FINAL EVALUATION ====================")

    eval_ckpt = best_path if os.path.exists(best_path) else last_path
    print(f"[EVAL] loading checkpoint: {eval_ckpt}")

    eval_model = YOLO(eval_ckpt)
    eval_model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore
    eval_model.model.to(device)  # type: ignore

    val_loss_df = valid_loss_only(
        eval_model,
        val_loader,
        device,
        epoch=best_epoch if os.path.exists(best_path) else cfg.epochs,
        max_val_steps=cfg.max_val_steps,
    )
    val_csv = os.path.join(cfg.checkpoint_dir, "val_losses.csv")
    val_loss_df.to_csv(val_csv, index=False)
    if len(val_loss_df) > 0:
        print("[FINAL VALID-LOSS] mean:", val_loss_df[["box", "cls", "dfl"]].mean().to_dict())
    else:
        print("[FINAL VALID-LOSS] no valid batches (empty labels?)")
    print(f"[INFO] saved val losses: {val_csv}")

    try:
        print("[FINAL VALID-METRICS] running ultralytics val() ...")
        device_id = 0 if device.type == "cuda" else "cpu"
        metrics = yolo_val_metrics(eval_model, cfg.data_yaml, device_id=device_id, workers=cfg.num_workers)

        print(f"Precision: {metrics.box.mp:.4f}")
        print(f"Recall:    {metrics.box.mr:.4f}")
        print(f"mAP@0.5:   {metrics.box.map50:.4f}")
        print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
    except Exception as e:
        print(f"[WARN] final model.val() failed: {e}")


if __name__ == "__main__":

    def running_in_notebook():
        try:
            from IPython import get_ipython

            return get_ipython() is not None
        except Exception:
            return False

    if running_in_notebook():
        print("[INFO] Running in Kaggle/Colab Notebook → using hardcoded config")
        cfg = Config(
            base_path="/kaggle/input/aod-4-yolo",
            data_yaml="/kaggle/working/data.yaml",
            epochs=4,
            batch_size=4,
            lr=3e-4,
            num_workers=2,  #  try 2; set 0 if you see worker issues
            max_train_steps=1000,
            max_val_steps=0,
        )
        main(cfg)
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument("--base_path", type=str, required=True)
        parser.add_argument("--data_yaml", type=str, required=True)
        parser.add_argument("--epochs", type=int, default=2)
        parser.add_argument("--batch_size", type=int, default=4)
        parser.add_argument("--lr", type=float, default=3e-4)
        parser.add_argument("--num_workers", type=int, default=2)
        parser.add_argument("--img_ext", type=str, default="jpg")
        parser.add_argument("--label_ext", type=str, default="txt")
        parser.add_argument("--checkpoint_dir", type=str, default="checkpoints")
        parser.add_argument("--max_train_steps", type=int, default=0)
        parser.add_argument("--max_val_steps", type=int, default=0)

        args = parser.parse_args()
        cfg = Config(**vars(args))
        main(cfg)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[INFO] Running in Kaggle/Colab Notebook → using hardcoded config
[INFO] device = cuda:0
[INFO] CUDA available: True
[INFO] CUDA device: Tesla P100-PCIE-16GB

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv     

train epoch 1:   0%|          | 0/3941 [00:00<?, ?it/s]

[DEBUG] images device: cuda:0
[DEBUG] targets device: cuda:0
[DEBUG] model device: cuda:0
[EPOCH 1] train_score=15.152403 | best_score=inf
[BEST] updated → epoch 1, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_1.pt (and last.pt)

==================== EPOCH 2/4 ====================


train epoch 2:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 2] train_score=11.202834 | best_score=15.152403
[BEST] updated → epoch 2, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_2.pt (and last.pt)

==================== EPOCH 3/4 ====================


...

[EPOCH 3] train_score=10.064760 | best_score=11.202834
[BEST] updated → epoch 3, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_3.pt (and last.pt)

==================== EPOCH 4/4 ====================


train epoch 4:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 4] train_score=9.562310 | best_score=10.064760
[BEST] updated → epoch 4, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_4.pt (and last.pt)
[INFO] saved train logs: checkpoints/train_losses.csv

==================== TRAIN SUMMARY ====================
[BEST MODEL] epoch = 4
[BEST MODEL] score = 9.562310
[BEST MODEL] path  = checkpoints/best.pt
[LAST MODEL] path  = checkpoints/last.pt

==================== FINAL EVALUATION ====================
[EVAL] loading checkpoint: checkpoints/best.pt


valid(loss) epoch 4:   0%|          | 0/1129 [00:00<?, ?it/s]

[FINAL VALID-LOSS] mean: {'box': 4.051972599414432, 'cls': 2.620748569612546, 'dfl': 3.4781040239761762}
[INFO] saved val losses: checkpoints/val_losses.csv
[FINAL VALID-METRICS] running ultralytics val() ...
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
custom_YOLOv11n summary (fused): 101 layers, 2,582,932 parameters, 13,260 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.2±0.4 ms, read: 48.1±14.9 MB/s, size: 46.9 KB)
val: Scanning /kaggle/input/aod-4-yolo/valid/labels... 4514 images, 125 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4514/4514 652.0it/s 6.9s<0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/aod-4-yolo/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 283/283 11.2it/s 25.2s<0.1s
                   all       4514       6369      0.214      0.246      0.147     0.0591
Speed: 0.8ms preprocess, 2.2ms inference, 0.0ms loss, 0

In [6]:
%%writefile hoangtn_CoordConv.py

import torch
import torch.nn as nn
from ultralytics.nn.modules.conv import Conv

class CoordConv(nn.Module):
    """
    CoordConv block (drop-in replacement for Ultralytics Conv)
    - Adds 2 coordinate channels (x, y) then applies a Conv.
    - x,y are normalized to [-1, 1] by default.
    """
    def __init__(
        self,
        c1: int,          # input channels
        c2: int,          # output channels
        k: int = 1,       # kernel
        s: int = 1,       # stride
        p=None,           # padding (None -> Ultralytics autopad inside Conv)
        g: int = 1,       # groups
        d: int = 1,       # dilation
        act=True,         # activation
        normalize: bool = True,  # normalize coords to [-1,1]
    ):
        super().__init__()
        self.normalize = normalize
        # Note: +2 channels for (x_coord, y_coord)
        self.conv = Conv(c1 + 2, c2, k, s, p, g, d, act)

    @staticmethod
    def _make_coord_grid(h: int, w: int, device, dtype, normalize: bool):
        # Create meshgrid (y, x) with shape [H,W]
        if normalize:
            xs = torch.linspace(-1.0, 1.0, w, device=device, dtype=dtype)
            ys = torch.linspace(-1.0, 1.0, h, device=device, dtype=dtype)
        else:
            xs = torch.arange(w, device=device, dtype=dtype)
            ys = torch.arange(h, device=device, dtype=dtype)

        yy, xx = torch.meshgrid(ys, xs, indexing="ij")  # [H,W], [H,W]
        # Return as [1,2,H,W]
        coord = torch.stack([xx, yy], dim=0).unsqueeze(0)
        return coord

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        coord = self._make_coord_grid(h, w, x.device, x.dtype, self.normalize)
        if coord.shape[0] != b:
            coord = coord.repeat(b, 1, 1, 1)  # [B,2,H,W]
        x = torch.cat([x, coord], dim=1)      # [B,C+2,H,W]
        return self.conv(x)

Writing hoangtn_CoordConv.py


In [7]:
# train_loop.py
import argparse
from dataclasses import dataclass

import pandas as pd
import torch
import torch.nn as nn

############################################################
# from hoangtn import HoangTN
from hoangtn_CoordConv import CoordConv
from torch import Tensor
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset

from ultralytics import YOLO

############################################################


# -----------------------------
# Utils
# -----------------------------
def get_device() -> torch.device:
    #  Force cuda:0 on Kaggle when available
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


# -----------------------------
# Dataset
# -----------------------------
class AodDataset(Dataset):
    """
    Read images + YOLO labels (cls xc yc w h) normalized [0..1].

    Return:
      - image: uint8 tensor [C,H,W] on CPU
      - targets: float tensor [N,5] on CPU (or [[-1]*5] if empty/missing).
    """

    def __init__(
        self,
        images_dir: str,
        labels_dir: str,
        images_ext: str = "jpg",
        labels_ext: str = "txt",
        transforms=None,
        verbose_bad_labels: bool = False,
    ):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.images_ext = images_ext
        self.labels_ext = labels_ext
        self.transforms = transforms
        self.verbose_bad_labels = verbose_bad_labels

        img_glob = os.path.join(images_dir, f"*.{images_ext}")
        self.images = sorted(glob.glob(img_glob))
        if len(self.images) == 0:
            raise FileNotFoundError(f"No images found: {img_glob}")

        self.miss_labeled: list[str] = []

    def _labels_path_from_image(self, image_path: str) -> str:
        base = os.path.basename(image_path)
        stem = base[: -(len(self.images_ext) + 1)]
        return os.path.join(self.labels_dir, f"{stem}.{self.labels_ext}")

    def _read_labels(self, labels_path: str) -> Tensor:
        if not os.path.exists(labels_path):
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        labels: list[list[float]] = []
        with open(labels_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) != 5:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid values count: {line}")
                    continue
                try:
                    c, xc, yc, w, h = map(float, parts)
                except ValueError:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: cannot parse: {line}")
                    continue

                if not (0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid bbox: {line}")
                    continue

                labels.append([c, xc, yc, w, h])

        if len(labels) == 0:
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        return torch.tensor(labels, dtype=torch.float32)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        image_path = self.images[idx]
        img = decode_image(image_path)  # uint8 [C,H,W] on CPU
        if self.transforms is not None:
            img = self.transforms(img)

        labels_path = self._labels_path_from_image(image_path)
        targets = self._read_labels(labels_path)
        return img, targets

    def __len__(self) -> int:
        return len(self.images)


# -----------------------------
# Collate & Batch builder
# -----------------------------
def collate_fn(batch: list[tuple[Tensor, Tensor]]) -> tuple[Tensor, Tensor]:
    images, targets = zip(*batch)
    images = torch.stack(images, dim=0)
    padded_targets = pad_sequence(list(targets), batch_first=True, padding_value=-1.0)
    return images, padded_targets


def prepare_batch_dict(images: Tensor, targets: Tensor) -> dict:
    valid_mask = targets[..., 0] != -1
    if not valid_mask.any():
        return {
            "img": images,
            "batch_idx": torch.empty((0,), device=images.device, dtype=torch.int64),
            "cls": torch.empty((0,), device=images.device, dtype=torch.float32),
            "bboxes": torch.empty((0, 4), device=images.device, dtype=torch.float32),
        }

    valid_targets = targets[valid_mask]  # [M,5]

    per_img_counts = valid_mask.sum(dim=1).to(torch.int64)
    batch_idx_list = []
    for i, cnt in enumerate(per_img_counts.tolist()):
        if cnt > 0:
            batch_idx_list.append(torch.full((cnt,), i, device=images.device, dtype=torch.int64))
    batch_idx = (
        torch.cat(batch_idx_list, dim=0)
        if batch_idx_list
        else torch.empty((0,), device=images.device, dtype=torch.int64)
    )

    cls = valid_targets[:, 0].to(torch.float32)
    bboxes = valid_targets[:, 1:].to(torch.float32)
    return {"img": images, "batch_idx": batch_idx, "cls": cls, "bboxes": bboxes}


# -----------------------------
# Train / Valid
# -----------------------------
def train_one_epoch(
    model: YOLO,
    optimizer: Adam,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_train_steps: int = 0,
    debug_print_first_batch: bool = True,
) -> pd.DataFrame:
    model.model.train(True)
    torch.set_grad_enabled(True)
    loss_fn = v8DetectionLoss(model.model)

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"train epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_train_steps > 0 and step >= max_train_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        if debug_print_first_batch and step == 0:
            print("[DEBUG] images device:", images.device)
            print("[DEBUG] targets device:", targets.device)
            print("[DEBUG] model device:", next(model.model.parameters()).device)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            pbar.set_postfix_str("skip(empty)")
            continue

        optimizer.zero_grad(set_to_none=True)

        #  Prevent inference_mode leakage from Ultralytics internals in notebooks
        with torch.inference_mode(False):
            with torch.enable_grad():
                preds = model.model(images)
                batch_loss, last_loss = loss_fn(preds, batch)
                loss_scalar = batch_loss.sum()

        if not torch.isfinite(loss_scalar):
            pbar.set_postfix_str("skip(non-finite)")
            continue

        if not loss_scalar.requires_grad:
            raise RuntimeError(
                f"loss_scalar.requires_grad=False at step {step}. "
                f"Grad is disabled globally or inference_mode is active."
            )

        loss_scalar.backward()
        optimizer.step()

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        row = {
            "epoch": epoch,
            "step": step,
            "box": float(box_loss),
            "cls": float(cls_loss),
            "dfl": float(dfl_loss),
        }
        rows.append(row)
        pbar.set_postfix({"box": f"{row['box']:.4f}", "cls": f"{row['cls']:.4f}", "dfl": f"{row['dfl']:.4f}"})

    return pd.DataFrame(rows)


@torch.no_grad()
def valid_loss_only(
    model: YOLO,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_val_steps: int = 0,
) -> pd.DataFrame:
    model.model.eval()
    loss_fn = v8DetectionLoss(model.model)

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"valid(loss) epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_val_steps > 0 and step >= max_val_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            continue

        preds = model.model(images)
        _, last_loss = loss_fn(preds, batch)

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        rows.append(
            {"epoch": epoch, "batch": step, "box": float(box_loss), "cls": float(cls_loss), "dfl": float(dfl_loss)}
        )

    return pd.DataFrame(rows)


def yolo_val_metrics(model: YOLO, data_yaml: str, device_id=0, workers: int = 0):
    # plots=False để tránh warning matplotlib & nhẹ hơn
    return model.val(data=data_yaml, device=device_id, workers=workers, verbose=False, plots=False)


# -----------------------------
# Main
# -----------------------------
@dataclass
class Config:
    base_path: str
    data_yaml: str
    epochs: int = 2
    batch_size: int = 4
    lr: float = 3e-4
    num_workers: int = 2
    img_ext: str = "jpg"
    label_ext: str = "txt"
    checkpoint_dir: str = "checkpoints"

    # Debug helpers (0 means "no limit")
    max_train_steps: int = 0
    max_val_steps: int = 0


def main(cfg: Config) -> None:
    device = get_device()
    print(f"[INFO] device = {device}")
    if device.type == "cuda":
        print("[INFO] CUDA available:", torch.cuda.is_available())
        print("[INFO] CUDA device:", torch.cuda.get_device_name(0))

    ensure_dir(cfg.checkpoint_dir)

    # model = YOLO(model="custom_yolov11n.yaml", task="detect", verbose=True)
    # model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore
    # model.model.to(device)  # type: ignore

    ####################################################################################################################################
    ######################################################## Swap Layer ################################################################
    # Swap CoordConv vào P2/4 và P3/8, bỏ swap HoangTN_CBAM
    ####################################################################################################################################

    # 1) load YOLO wrapper
    model = YOLO(model="custom_yolov11n.yaml", task="detect", verbose=True)

    # 2) set loss weights
    model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore

    # 3) SWAP: Conv stride=2 (P2/4 out=128) và (P3/8 out=256) -> CoordConv
    layers = model.model.model  # nn.ModuleList các layer

    def is_ultra_conv(m: nn.Module) -> bool:
        # Ultralytics Conv wrapper thường có attribute .conv (nn.Conv2d)
        return hasattr(m, "conv") and isinstance(getattr(m, "conv", None), nn.Conv2d)

    def get_conv_meta(m: nn.Module):
        # Trả về (in_ch, out_ch, stride) nếu lấy được, else None
        if is_ultra_conv(m):
            conv = m.conv
            # stride có thể là tuple (sH,sW)
            s = conv.stride[0] if isinstance(conv.stride, tuple) else int(conv.stride)
            return conv.in_channels, conv.out_channels, s
        return None

    targets = {"p2": None, "p3": None}  # store indices

    # Tìm Conv stride=2 có out=128 (P2/4) và out=256 (P3/8)
    for i, m in enumerate(layers):
        meta = get_conv_meta(m)
        if meta is None:
            continue
        c1, c2, s = meta
        if s == 2 and c2 == 128 and targets["p2"] is None:
            targets["p2"] = i
        if s == 2 and c2 == 256 and targets["p3"] is None:
            targets["p3"] = i

    # Nếu không tìm thấy, in debug tất cả Conv stride=2 để bạn đối chiếu
    if targets["p2"] is None or targets["p3"] is None:
        print("\n[DEBUG] List Conv(stride=2):")
        for i, m in enumerate(layers):
            meta = get_conv_meta(m)
            if meta is None:
                continue
            c1, c2, s = meta
            if s == 2:
                print(f"  idx={i:3d}  Conv  {c1}->{c2}  stride={s}")
        raise RuntimeError(f"Không tìm thấy Conv stride=2 cho P2/4(out=128) hoặc P3/8(out=256). Found={targets}")

    # Swap P2/4
    idx_p2 = targets["p2"]
    old_p2 = layers[idx_p2]
    c1_p2, c2_p2, _s_p2 = get_conv_meta(old_p2)
    new_p2 = CoordConv(c1_p2, c2_p2, k=3, s=2, normalize=True)

    # Copy Ultralytics metadata
    for k in ("f", "i", "type", "np"):
        if hasattr(old_p2, k):
            setattr(new_p2, k, getattr(old_p2, k))
    new_p2.training = old_p2.training
    layers[idx_p2] = new_p2
    print(f"[SWAP OK] P2/4 at idx={idx_p2}: Conv({c1_p2}->{c2_p2}, s=2) -> CoordConv")

    # Swap P3/8
    idx_p3 = targets["p3"]
    old_p3 = layers[idx_p3]
    c1_p3, c2_p3, _s_p3 = get_conv_meta(old_p3)
    new_p3 = CoordConv(c1_p3, c2_p3, k=3, s=2, normalize=True)

    for k in ("f", "i", "type", "np"):
        if hasattr(old_p3, k):
            setattr(new_p3, k, getattr(old_p3, k))
    new_p3.training = old_p3.training
    layers[idx_p3] = new_p3
    print(f"[SWAP OK] P3/8 at idx={idx_p3}: Conv({c1_p3}->{c2_p3}, s=2) -> CoordConv")

    # 4) move to device
    model.model.to(device)  # type: ignore
    ####################################################################################################################################
    ####################################################################################################################################
    print("\n================ MODEL INFO (after swap) ================\n", flush=True)
    model.model.info()

    print("\n================ LAYERS (after swap) ================\n", flush=True)

    swapped_idxs = {idx_p2, idx_p3}

    for i, m in enumerate(model.model.model):
        name = m.__class__.__name__
        f = getattr(m, "f", None)

        np_ = sum(p.numel() for p in m.parameters()) if isinstance(m, nn.Module) else 0

        c1 = c2 = None
        if hasattr(m, "cv1") and hasattr(m, "cv2") and hasattr(m.cv1, "conv") and hasattr(m.cv2, "conv"):
            try:
                c1 = m.cv1.conv.in_channels
                c2 = m.cv2.conv.out_channels
            except Exception:
                pass

        mark = "  <== SWAPPED" if i in swapped_idxs else ""

        if c1 is not None and c2 is not None:
            print(f"{i:3d}  f={f!s:>6}  {name:<20}  c1->c2: {c1:4d}->{c2:<4d}  params={np_:>8d}{mark}")
        else:
            print(f"{i:3d}  f={f!s:>6}  {name:<20}  params={np_:>8d}{mark}")

    ####################################################################################################################################

    print("[DEBUG] model param device:", next(model.model.parameters()).device)

    optimizer = Adam(model.model.parameters(), lr=cfg.lr)

    train_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "train/images"),
        labels_dir=os.path.join(cfg.base_path, "train/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )
    val_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "valid/images"),
        labels_dir=os.path.join(cfg.base_path, "valid/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )

    pin = torch.cuda.is_available()
    workers = cfg.num_workers

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )

    train_all = []

    best_score = float("inf")
    best_epoch = -1
    best_path = os.path.join(cfg.checkpoint_dir, "best.pt")
    last_path = os.path.join(cfg.checkpoint_dir, "last.pt")

    for epoch in range(1, cfg.epochs + 1):
        print(f"\n==================== EPOCH {epoch}/{cfg.epochs} ====================")

        train_df = train_one_epoch(
            model,
            optimizer,
            train_loader,
            device,
            epoch,
            max_train_steps=cfg.max_train_steps,
            debug_print_first_batch=(epoch == 1),
        )
        train_all.append(train_df)

        if len(train_df) > 0:
            train_score = train_df[["box", "cls", "dfl"]].mean().sum()
            print(f"[EPOCH {epoch}] train_score={train_score:.6f} | best_score={best_score:.6f}")

            if train_score < best_score:
                best_score = train_score
                best_epoch = epoch
                model.save(best_path)
                print(f"[BEST] updated → epoch {best_epoch}, saved to {best_path}")
        else:
            print("[TRAIN] no valid batches produced loss (all empty labels?)")

        model.save(last_path)
        ckpt_path = os.path.join(cfg.checkpoint_dir, f"epoch_{epoch}.pt")
        model.save(ckpt_path)
        print(f"[INFO] saved checkpoint: {ckpt_path} (and last.pt)")

    train_csv = os.path.join(cfg.checkpoint_dir, "train_losses.csv")
    pd.concat(train_all, axis=0, ignore_index=True).to_csv(train_csv, index=False)
    print(f"[INFO] saved train logs: {train_csv}")

    print("\n==================== TRAIN SUMMARY ====================")
    print(f"[BEST MODEL] epoch = {best_epoch}")
    print(f"[BEST MODEL] score = {best_score:.6f}")
    print(f"[BEST MODEL] path  = {best_path}")
    print(f"[LAST MODEL] path  = {last_path}")

    print("\n==================== FINAL EVALUATION ====================")

    eval_ckpt = best_path if os.path.exists(best_path) else last_path
    print(f"[EVAL] loading checkpoint: {eval_ckpt}")

    eval_model = YOLO(eval_ckpt)
    eval_model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore
    eval_model.model.to(device)  # type: ignore

    val_loss_df = valid_loss_only(
        eval_model,
        val_loader,
        device,
        epoch=best_epoch if os.path.exists(best_path) else cfg.epochs,
        max_val_steps=cfg.max_val_steps,
    )
    val_csv = os.path.join(cfg.checkpoint_dir, "val_losses.csv")
    val_loss_df.to_csv(val_csv, index=False)
    if len(val_loss_df) > 0:
        print("[FINAL VALID-LOSS] mean:", val_loss_df[["box", "cls", "dfl"]].mean().to_dict())
    else:
        print("[FINAL VALID-LOSS] no valid batches (empty labels?)")
    print(f"[INFO] saved val losses: {val_csv}")

    try:
        print("[FINAL VALID-METRICS] running ultralytics val() ...")
        device_id = 0 if device.type == "cuda" else "cpu"
        metrics = yolo_val_metrics(eval_model, cfg.data_yaml, device_id=device_id, workers=cfg.num_workers)

        print(f"Precision: {metrics.box.mp:.4f}")
        print(f"Recall:    {metrics.box.mr:.4f}")
        print(f"mAP@0.5:   {metrics.box.map50:.4f}")
        print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
    except Exception as e:
        print(f"[WARN] final model.val() failed: {e}")


if __name__ == "__main__":

    def running_in_notebook():
        try:
            from IPython import get_ipython

            return get_ipython() is not None
        except Exception:
            return False

    if running_in_notebook():
        print("[INFO] Running in Kaggle/Colab Notebook → using hardcoded config")
        cfg = Config(
            base_path="/kaggle/input/aod-4-yolo",
            data_yaml="/kaggle/working/data.yaml",
            epochs=4,
            batch_size=4,
            lr=3e-4,
            num_workers=2,  # ✅ try 2; set 0 if you see worker issues
            max_train_steps=1000,
            max_val_steps=0,
        )
        main(cfg)
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument("--base_path", type=str, required=True)
        parser.add_argument("--data_yaml", type=str, required=True)
        parser.add_argument("--epochs", type=int, default=2)
        parser.add_argument("--batch_size", type=int, default=4)
        parser.add_argument("--lr", type=float, default=3e-4)
        parser.add_argument("--num_workers", type=int, default=2)
        parser.add_argument("--img_ext", type=str, default="jpg")
        parser.add_argument("--label_ext", type=str, default="txt")
        parser.add_argument("--checkpoint_dir", type=str, default="checkpoints")
        parser.add_argument("--max_train_steps", type=int, default=0)
        parser.add_argument("--max_val_steps", type=int, default=0)

        args = parser.parse_args()
        cfg = Config(**vars(args))
        main(cfg)

[INFO] Running in Kaggle/Colab Notebook → using hardcoded config
[INFO] device = cuda:0
[INFO] CUDA available: True
[INFO] CUDA device: Tesla P100-PCIE-16GB

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultral

train epoch 1:   0%|          | 0/3941 [00:00<?, ?it/s]

[DEBUG] images device: cuda:0
[DEBUG] targets device: cuda:0
[DEBUG] model device: cuda:0
[EPOCH 1] train_score=15.122460 | best_score=inf
[BEST] updated → epoch 1, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_1.pt (and last.pt)

==================== EPOCH 2/4 ====================


train epoch 2:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 2] train_score=11.262545 | best_score=15.122460
[BEST] updated → epoch 2, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_2.pt (and last.pt)

==================== EPOCH 3/4 ====================


...

[EPOCH 3] train_score=10.233939 | best_score=11.262545
[BEST] updated → epoch 3, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_3.pt (and last.pt)

==================== EPOCH 4/4 ====================


train epoch 4:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 4] train_score=9.717857 | best_score=10.233939
[BEST] updated → epoch 4, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_4.pt (and last.pt)
[INFO] saved train logs: checkpoints/train_losses.csv

==================== TRAIN SUMMARY ====================
[BEST MODEL] epoch = 4
[BEST MODEL] score = 9.717857
[BEST MODEL] path  = checkpoints/best.pt
[LAST MODEL] path  = checkpoints/last.pt

==================== FINAL EVALUATION ====================
[EVAL] loading checkpoint: checkpoints/best.pt


valid(loss) epoch 4:   0%|          | 0/1129 [00:00<?, ?it/s]

[FINAL VALID-LOSS] mean: {'box': 4.142463504038584, 'cls': 2.724397389076216, 'dfl': 3.592014027604073}
[INFO] saved val losses: checkpoints/val_losses.csv
[FINAL VALID-METRICS] running ultralytics val() ...
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
custom_YOLOv11n summary (fused): 101 layers, 2,589,844 parameters, 13,260 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 36.8±12.2 MB/s, size: 38.1 KB)
val: Scanning /kaggle/input/aod-4-yolo/valid/labels... 4514 images, 125 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4514/4514 677.0it/s 6.7s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/aod-4-yolo/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 283/283 11.3it/s 25.1s0.2ss
                   all       4514       6369      0.211      0.271      0.138     0.0482
Speed: 0.8ms preprocess, 2.3ms inference, 0.0ms loss, 0.9

Swap CoordConv to MAPBoostLite

In [8]:
%%writefile hoangtn_MAPBoostLite.py

import torch
import torch.nn as nn

class CoordConv2d(nn.Module):
    """
    CoordConv2d (core): Adds 2 coordinate channels (x, y) then applies Conv2d.
    - x,y normalized to [-1, 1] by default.
    - Uses raw nn.Conv2d (NO BN/Act) so MAPBoostLite can add BN+SiLU explicitly (paper-correct).
    """
    def __init__(
        self,
        c1: int,
        c2: int,
        k: int = 3,
        s: int = 1,
        p: int | None = None,
        g: int = 1,
        d: int = 1,
        bias: bool = False,
        normalize: bool = True,
    ):
        super().__init__()
        self.normalize = normalize
        if p is None:
            # autopad "same" for odd kernels (Ultralytics-style)
            p = (k - 1) // 2 * d

        self.conv = nn.Conv2d(
            in_channels=c1 + 2,
            out_channels=c2,
            kernel_size=k,
            stride=s,
            padding=p,
            dilation=d,
            groups=g,
            bias=bias,
        )

    @staticmethod
    def _make_coord_grid(h: int, w: int, device, dtype, normalize: bool):
        if normalize:
            xs = torch.linspace(-1.0, 1.0, w, device=device, dtype=dtype)
            ys = torch.linspace(-1.0, 1.0, h, device=device, dtype=dtype)
        else:
            xs = torch.arange(w, device=device, dtype=dtype)
            ys = torch.arange(h, device=device, dtype=dtype)

        yy, xx = torch.meshgrid(ys, xs, indexing="ij")  # [H,W]
        coord = torch.stack([xx, yy], dim=0).unsqueeze(0)  # [1,2,H,W]
        return coord

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, _, h, w = x.shape
        coord = self._make_coord_grid(h, w, x.device, x.dtype, self.normalize)
        if coord.shape[0] != b:
            coord = coord.repeat(b, 1, 1, 1)  # [B,2,H,W]
        x = torch.cat([x, coord], dim=1)  # [B,C+2,H,W]
        return self.conv(x)


class MAPBoostLite(nn.Module):
    """
    MAPBoostLite (paper): CoordConv2d + BN + SiLU
    Drop-in thay cho CoordConv/Conv ở backbone sớm.
    """
    def __init__(
        self,
        c1: int,
        c2: int,
        k: int = 3,
        s: int = 1,
        p: int | None = None,
        g: int = 1,
        d: int = 1,
        normalize: bool = True,
    ):
        super().__init__()
        self.coord = CoordConv2d(c1, c2, k=k, s=s, p=p, g=g, d=d, bias=False, normalize=normalize)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.coord(x)
        x = self.bn(x)
        x = self.act(x)
        return x


Writing hoangtn_MAPBoostLite.py


In [14]:
# train_loop.py
import argparse
from dataclasses import dataclass

import pandas as pd
import torch

############################################################
# from hoangtn import HoangTN
from hoangtn_MAPBoostLite import MAPBoostLite
from torch import Tensor
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset

from ultralytics import YOLO

############################################################


# -----------------------------
# Utils
# -----------------------------
def get_device() -> torch.device:
    #  Force cuda:0 on Kaggle when available
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


# -----------------------------
# Dataset
# -----------------------------
class AodDataset(Dataset):
    """
    Read images + YOLO labels (cls xc yc w h) normalized [0..1].

    Return:
      - image: uint8 tensor [C,H,W] on CPU
      - targets: float tensor [N,5] on CPU (or [[-1]*5] if empty/missing).
    """

    def __init__(
        self,
        images_dir: str,
        labels_dir: str,
        images_ext: str = "jpg",
        labels_ext: str = "txt",
        transforms=None,
        verbose_bad_labels: bool = False,
    ):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.images_ext = images_ext
        self.labels_ext = labels_ext
        self.transforms = transforms
        self.verbose_bad_labels = verbose_bad_labels

        img_glob = os.path.join(images_dir, f"*.{images_ext}")
        self.images = sorted(glob.glob(img_glob))
        if len(self.images) == 0:
            raise FileNotFoundError(f"No images found: {img_glob}")

        self.miss_labeled: list[str] = []

    def _labels_path_from_image(self, image_path: str) -> str:
        base = os.path.basename(image_path)
        stem = base[: -(len(self.images_ext) + 1)]
        return os.path.join(self.labels_dir, f"{stem}.{self.labels_ext}")

    def _read_labels(self, labels_path: str) -> Tensor:
        if not os.path.exists(labels_path):
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        labels: list[list[float]] = []
        with open(labels_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) != 5:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid values count: {line}")
                    continue
                try:
                    c, xc, yc, w, h = map(float, parts)
                except ValueError:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: cannot parse: {line}")
                    continue

                if not (0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid bbox: {line}")
                    continue

                labels.append([c, xc, yc, w, h])

        if len(labels) == 0:
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        return torch.tensor(labels, dtype=torch.float32)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        image_path = self.images[idx]
        img = decode_image(image_path)  # uint8 [C,H,W] on CPU
        if self.transforms is not None:
            img = self.transforms(img)

        labels_path = self._labels_path_from_image(image_path)
        targets = self._read_labels(labels_path)
        return img, targets

    def __len__(self) -> int:
        return len(self.images)


# -----------------------------
# Collate & Batch builder
# -----------------------------
def collate_fn(batch: list[tuple[Tensor, Tensor]]) -> tuple[Tensor, Tensor]:
    images, targets = zip(*batch)
    images = torch.stack(images, dim=0)
    padded_targets = pad_sequence(list(targets), batch_first=True, padding_value=-1.0)
    return images, padded_targets


def prepare_batch_dict(images: Tensor, targets: Tensor) -> dict:
    valid_mask = targets[..., 0] != -1
    if not valid_mask.any():
        return {
            "img": images,
            "batch_idx": torch.empty((0,), device=images.device, dtype=torch.int64),
            "cls": torch.empty((0,), device=images.device, dtype=torch.float32),
            "bboxes": torch.empty((0, 4), device=images.device, dtype=torch.float32),
        }

    valid_targets = targets[valid_mask]  # [M,5]

    per_img_counts = valid_mask.sum(dim=1).to(torch.int64)
    batch_idx_list = []
    for i, cnt in enumerate(per_img_counts.tolist()):
        if cnt > 0:
            batch_idx_list.append(torch.full((cnt,), i, device=images.device, dtype=torch.int64))
    batch_idx = (
        torch.cat(batch_idx_list, dim=0)
        if batch_idx_list
        else torch.empty((0,), device=images.device, dtype=torch.int64)
    )

    cls = valid_targets[:, 0].to(torch.float32)
    bboxes = valid_targets[:, 1:].to(torch.float32)
    return {"img": images, "batch_idx": batch_idx, "cls": cls, "bboxes": bboxes}


# -----------------------------
# Train / Valid
# -----------------------------
def train_one_epoch(
    model: YOLO,
    optimizer: Adam,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_train_steps: int = 0,
    debug_print_first_batch: bool = True,
) -> pd.DataFrame:
    model.model.train(True)
    torch.set_grad_enabled(True)
    loss_fn = v8DetectionLoss(model.model)

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"train epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_train_steps > 0 and step >= max_train_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        if debug_print_first_batch and step == 0:
            print("[DEBUG] images device:", images.device)
            print("[DEBUG] targets device:", targets.device)
            print("[DEBUG] model device:", next(model.model.parameters()).device)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            pbar.set_postfix_str("skip(empty)")
            continue

        optimizer.zero_grad(set_to_none=True)

        #  Prevent inference_mode leakage from Ultralytics internals in notebooks
        with torch.inference_mode(False):
            with torch.enable_grad():
                preds = model.model(images)
                batch_loss, last_loss = loss_fn(preds, batch)
                loss_scalar = batch_loss.sum()

        if not torch.isfinite(loss_scalar):
            pbar.set_postfix_str("skip(non-finite)")
            continue

        if not loss_scalar.requires_grad:
            raise RuntimeError(
                f"loss_scalar.requires_grad=False at step {step}. "
                f"Grad is disabled globally or inference_mode is active."
            )

        loss_scalar.backward()
        optimizer.step()

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        row = {
            "epoch": epoch,
            "step": step,
            "box": float(box_loss),
            "cls": float(cls_loss),
            "dfl": float(dfl_loss),
        }
        rows.append(row)
        pbar.set_postfix({"box": f"{row['box']:.4f}", "cls": f"{row['cls']:.4f}", "dfl": f"{row['dfl']:.4f}"})

    return pd.DataFrame(rows)


@torch.no_grad()
def valid_loss_only(
    model: YOLO,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_val_steps: int = 0,
) -> pd.DataFrame:
    model.model.eval()
    loss_fn = v8DetectionLoss(model.model)

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"valid(loss) epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_val_steps > 0 and step >= max_val_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            continue

        preds = model.model(images)
        _, last_loss = loss_fn(preds, batch)

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        rows.append(
            {"epoch": epoch, "batch": step, "box": float(box_loss), "cls": float(cls_loss), "dfl": float(dfl_loss)}
        )

    return pd.DataFrame(rows)


def yolo_val_metrics(model: YOLO, data_yaml: str, device_id=0, workers: int = 0):
    # plots=False để tránh warning matplotlib & nhẹ hơn
    return model.val(data=data_yaml, device=device_id, workers=workers, verbose=False, plots=False)


# -----------------------------
# Main
# -----------------------------
@dataclass
class Config:
    base_path: str
    data_yaml: str
    epochs: int = 2
    batch_size: int = 4
    lr: float = 3e-4
    num_workers: int = 2
    img_ext: str = "jpg"
    label_ext: str = "txt"
    checkpoint_dir: str = "checkpoints"

    # Debug helpers (0 means "no limit")
    max_train_steps: int = 0
    max_val_steps: int = 0


def main(cfg: Config) -> None:
    device = get_device()
    print(f"[INFO] device = {device}")
    if device.type == "cuda":
        print("[INFO] CUDA available:", torch.cuda.is_available())
        print("[INFO] CUDA device:", torch.cuda.get_device_name(0))

    ensure_dir(cfg.checkpoint_dir)

    # model = YOLO(model="custom_yolov11n.yaml", task="detect", verbose=True)
    # model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore
    # model.model.to(device)  # type: ignore

    ####################################################################################################################################
    ######################################################## Swap Layer ################################################################
    # Swap MAPBoostLite
    ####################################################################################################################################

    # 1) load YOLO wrapper
    model = YOLO(model="custom_yolov11n.yaml", task="detect", verbose=True)

    # 2) set loss weights
    model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore

    # 3) SWAP: Conv stride=2 (P2/4 out=128) và (P3/8 out=256) -> CoordConv
    layers = model.model.model  # nn.ModuleList các layer

    def is_ultra_conv(m: nn.Module) -> bool:
        # Ultralytics Conv wrapper thường có attribute .conv (nn.Conv2d)
        return hasattr(m, "conv") and isinstance(getattr(m, "conv", None), nn.Conv2d)

    def get_conv_meta(m: nn.Module):
        # Trả về (in_ch, out_ch, stride) nếu lấy được, else None
        if is_ultra_conv(m):
            conv = m.conv
            # stride có thể là tuple (sH,sW)
            s = conv.stride[0] if isinstance(conv.stride, tuple) else int(conv.stride)
            return conv.in_channels, conv.out_channels, s
        return None

    targets = {"p2": None, "p3": None}  # store indices

    # Tìm Conv stride=2 có out=128 (P2/4) và out=256 (P3/8)
    for i, m in enumerate(layers):
        meta = get_conv_meta(m)
        if meta is None:
            continue
        c1, c2, s = meta
        if s == 2 and c2 == 128 and targets["p2"] is None:
            targets["p2"] = i
        if s == 2 and c2 == 256 and targets["p3"] is None:
            targets["p3"] = i

    # Nếu không tìm thấy, in debug tất cả Conv stride=2 để bạn đối chiếu
    if targets["p2"] is None or targets["p3"] is None:
        print("\n[DEBUG] List Conv(stride=2):")
        for i, m in enumerate(layers):
            meta = get_conv_meta(m)
            if meta is None:
                continue
            c1, c2, s = meta
            if s == 2:
                print(f"  idx={i:3d}  Conv  {c1}->{c2}  stride={s}")
        raise RuntimeError(f"Không tìm thấy Conv stride=2 cho P2/4(out=128) hoặc P3/8(out=256). Found={targets}")

    # Swap P2/4
    idx_p2 = targets["p2"]
    old_p2 = layers[idx_p2]
    c1_p2, c2_p2, _s_p2 = get_conv_meta(old_p2)
    new_p2 = MAPBoostLite(c1_p2, c2_p2, k=3, s=2, normalize=True)

    # Copy Ultralytics metadata
    for k in ("f", "i", "type", "np"):
        if hasattr(old_p2, k):
            setattr(new_p2, k, getattr(old_p2, k))
    new_p2.training = old_p2.training
    layers[idx_p2] = new_p2
    print(f"[SWAP OK] P2/4 at idx={idx_p2}: Conv({c1_p2}->{c2_p2}, s=2) -> MAPBoostLite")

    # Swap P3/8
    idx_p3 = targets["p3"]
    old_p3 = layers[idx_p3]
    c1_p3, c2_p3, _s_p3 = get_conv_meta(old_p3)
    new_p3 = MAPBoostLite(c1_p3, c2_p3, k=3, s=2, normalize=True)

    for k in ("f", "i", "type", "np"):
        if hasattr(old_p3, k):
            setattr(new_p3, k, getattr(old_p3, k))
    new_p3.training = old_p3.training
    layers[idx_p3] = new_p3
    print(f"[SWAP OK] P3/8 at idx={idx_p3}: Conv({c1_p3}->{c2_p3}, s=2) -> MAPBoostLite")

    # 4) move to device
    model.model.to(device)  # type: ignore
    ####################################################################################################################################
    ####################################################################################################################################
    print("\n================ MODEL INFO (after swap) ================\n", flush=True)
    model.model.info()

    print("\n================ LAYERS (after swap) ================\n", flush=True)

    swapped_idxs = {idx_p2, idx_p3}

    for i, m in enumerate(model.model.model):
        name = m.__class__.__name__
        f = getattr(m, "f", None)

        np_ = sum(p.numel() for p in m.parameters()) if isinstance(m, nn.Module) else 0

        c1 = c2 = None
        if hasattr(m, "cv1") and hasattr(m, "cv2") and hasattr(m.cv1, "conv") and hasattr(m.cv2, "conv"):
            try:
                c1 = m.cv1.conv.in_channels
                c2 = m.cv2.conv.out_channels
            except Exception:
                pass

        mark = "  <== SWAPPED" if i in swapped_idxs else ""

        if c1 is not None and c2 is not None:
            print(f"{i:3d}  f={f!s:>6}  {name:<20}  c1->c2: {c1:4d}->{c2:<4d}  params={np_:>8d}{mark}")
        else:
            print(f"{i:3d}  f={f!s:>6}  {name:<20}  params={np_:>8d}{mark}")

    ####################################################################################################################################

    print("[DEBUG] model param device:", next(model.model.parameters()).device)

    optimizer = Adam(model.model.parameters(), lr=cfg.lr)

    train_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "train/images"),
        labels_dir=os.path.join(cfg.base_path, "train/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )
    val_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "valid/images"),
        labels_dir=os.path.join(cfg.base_path, "valid/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )

    pin = torch.cuda.is_available()
    workers = cfg.num_workers

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )

    train_all = []

    best_score = float("inf")
    best_epoch = -1
    best_path = os.path.join(cfg.checkpoint_dir, "best.pt")
    last_path = os.path.join(cfg.checkpoint_dir, "last.pt")

    for epoch in range(1, cfg.epochs + 1):
        print(f"\n==================== EPOCH {epoch}/{cfg.epochs} ====================")

        train_df = train_one_epoch(
            model,
            optimizer,
            train_loader,
            device,
            epoch,
            max_train_steps=cfg.max_train_steps,
            debug_print_first_batch=(epoch == 1),
        )
        train_all.append(train_df)

        if len(train_df) > 0:
            train_score = train_df[["box", "cls", "dfl"]].mean().sum()
            print(f"[EPOCH {epoch}] train_score={train_score:.6f} | best_score={best_score:.6f}")

            if train_score < best_score:
                best_score = train_score
                best_epoch = epoch
                model.save(best_path)
                print(f"[BEST] updated → epoch {best_epoch}, saved to {best_path}")
        else:
            print("[TRAIN] no valid batches produced loss (all empty labels?)")

        model.save(last_path)
        ckpt_path = os.path.join(cfg.checkpoint_dir, f"epoch_{epoch}.pt")
        model.save(ckpt_path)
        print(f"[INFO] saved checkpoint: {ckpt_path} (and last.pt)")

    train_csv = os.path.join(cfg.checkpoint_dir, "train_losses.csv")
    pd.concat(train_all, axis=0, ignore_index=True).to_csv(train_csv, index=False)
    print(f"[INFO] saved train logs: {train_csv}")

    print("\n==================== TRAIN SUMMARY ====================")
    print(f"[BEST MODEL] epoch = {best_epoch}")
    print(f"[BEST MODEL] score = {best_score:.6f}")
    print(f"[BEST MODEL] path  = {best_path}")
    print(f"[LAST MODEL] path  = {last_path}")

    print("\n==================== FINAL EVALUATION ====================")

    eval_ckpt = best_path if os.path.exists(best_path) else last_path
    print(f"[EVAL] loading checkpoint: {eval_ckpt}")

    eval_model = YOLO(eval_ckpt)
    eval_model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore
    eval_model.model.to(device)  # type: ignore

    val_loss_df = valid_loss_only(
        eval_model,
        val_loader,
        device,
        epoch=best_epoch if os.path.exists(best_path) else cfg.epochs,
        max_val_steps=cfg.max_val_steps,
    )
    val_csv = os.path.join(cfg.checkpoint_dir, "val_losses.csv")
    val_loss_df.to_csv(val_csv, index=False)
    if len(val_loss_df) > 0:
        print("[FINAL VALID-LOSS] mean:", val_loss_df[["box", "cls", "dfl"]].mean().to_dict())
    else:
        print("[FINAL VALID-LOSS] no valid batches (empty labels?)")
    print(f"[INFO] saved val losses: {val_csv}")

    try:
        print("[FINAL VALID-METRICS] running ultralytics val() ...")
        device_id = 0 if device.type == "cuda" else "cpu"
        metrics = yolo_val_metrics(eval_model, cfg.data_yaml, device_id=device_id, workers=cfg.num_workers)

        print(f"Precision: {metrics.box.mp:.4f}")
        print(f"Recall:    {metrics.box.mr:.4f}")
        print(f"mAP@0.5:   {metrics.box.map50:.4f}")
        print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
    except Exception as e:
        print(f"[WARN] final model.val() failed: {e}")
    return model


if __name__ == "__main__":

    def running_in_notebook():
        try:
            from IPython import get_ipython

            return get_ipython() is not None
        except Exception:
            return False

    if running_in_notebook():
        print("[INFO] Running in Kaggle/Colab Notebook → using hardcoded config")
        cfg = Config(
            base_path="/kaggle/input/aod-4-yolo",
            data_yaml="/kaggle/working/data.yaml",
            epochs=4,
            batch_size=4,
            lr=3e-4,
            num_workers=2,  # ✅ try 2; set 0 if you see worker issues
            max_train_steps=1000,
            max_val_steps=0,
        )
        main(cfg)
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument("--base_path", type=str, required=True)
        parser.add_argument("--data_yaml", type=str, required=True)
        parser.add_argument("--epochs", type=int, default=2)
        parser.add_argument("--batch_size", type=int, default=4)
        parser.add_argument("--lr", type=float, default=3e-4)
        parser.add_argument("--num_workers", type=int, default=2)
        parser.add_argument("--img_ext", type=str, default="jpg")
        parser.add_argument("--label_ext", type=str, default="txt")
        parser.add_argument("--checkpoint_dir", type=str, default="checkpoints")
        parser.add_argument("--max_train_steps", type=int, default=0)
        parser.add_argument("--max_val_steps", type=int, default=0)

        args = parser.parse_args()
        cfg = Config(**vars(args))
        main(cfg)

[INFO] Running in Kaggle/Colab Notebook → using hardcoded config
[INFO] device = cuda:0
[INFO] CUDA available: True
[INFO] CUDA device: Tesla P100-PCIE-16GB

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultral

train epoch 1:   0%|          | 0/3941 [00:00<?, ?it/s]

[DEBUG] images device: cuda:0
[DEBUG] targets device: cuda:0
[DEBUG] model device: cuda:0
[EPOCH 1] train_score=15.228699 | best_score=inf
[BEST] updated → epoch 1, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_1.pt (and last.pt)

==================== EPOCH 2/4 ====================


train epoch 2:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 2] train_score=11.413248 | best_score=15.228699
[BEST] updated → epoch 2, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_2.pt (and last.pt)

==================== EPOCH 3/4 ====================


...

[EPOCH 3] train_score=10.490231 | best_score=11.413248
[BEST] updated → epoch 3, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_3.pt (and last.pt)

==================== EPOCH 4/4 ====================


train epoch 4:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 4] train_score=9.904334 | best_score=10.490231
[BEST] updated → epoch 4, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_4.pt (and last.pt)
[INFO] saved train logs: checkpoints/train_losses.csv

==================== TRAIN SUMMARY ====================
[BEST MODEL] epoch = 4
[BEST MODEL] score = 9.904334
[BEST MODEL] path  = checkpoints/best.pt
[LAST MODEL] path  = checkpoints/last.pt

==================== FINAL EVALUATION ====================
[EVAL] loading checkpoint: checkpoints/best.pt


valid(loss) epoch 4:   0%|          | 0/1129 [00:00<?, ?it/s]

[FINAL VALID-LOSS] mean: {'box': 4.132773268276266, 'cls': 2.575048176215903, 'dfl': 3.5925892755055107}
[INFO] saved val losses: checkpoints/val_losses.csv
[FINAL VALID-METRICS] running ultralytics val() ...
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
custom_YOLOv11n summary (fused): 105 layers, 2,590,228 parameters, 463,308 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 34.3±10.5 MB/s, size: 34.8 KB)
val: Scanning /kaggle/input/aod-4-yolo/valid/labels... 4514 images, 125 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4514/4514 894.3it/s 5.0s<0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/aod-4-yolo/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 283/283 10.9it/s 25.9s0.2ss
                   all       4514       6369      0.267      0.285      0.178     0.0648
Speed: 0.8ms preprocess, 2.2ms inference, 0.0ms loss, 

# Bắt đầu chỉ layer 16,19 -> MAPBoost full (SobelEdgePool): đặt ở layer 16 (bắt buộc), 19 là optional đo mAP@75 → rồi mới mở rộng.


# Bước 1 – (bạn đang ở đây) Giữ Detect (f=[16,19,22]) Add MAPBoost full tại layer 16,19


In [16]:
%%writefile hoangtn_MAPBoost.py
import torch
import torch.nn as nn
import torch.nn.functional as F


# =========================================================
# Core: CoordConv2d (raw Conv2d, no BN/Act)
# =========================================================
class CoordConv2d(nn.Module):
    """
    Adds 2 coordinate channels (x,y) then applies nn.Conv2d (no BN/Act).
    Coordinates are normalized to [-1, 1] by default.
    """
    def __init__(
        self,
        c1: int,
        c2: int,
        k: int = 3,
        s: int = 1,
        p: int | None = None,
        g: int = 1,
        d: int = 1,
        bias: bool = False,
        normalize: bool = True,
    ):
        super().__init__()
        self.normalize = normalize
        if p is None:
            p = (k - 1) // 2 * d  # autopad-like

        self.conv = nn.Conv2d(
            c1 + 2, c2,
            kernel_size=k,
            stride=s,
            padding=p,
            dilation=d,
            groups=g,
            bias=bias,
        )

    @staticmethod
    def _make_coord_grid(h: int, w: int, device, dtype, normalize: bool):
        if normalize:
            xs = torch.linspace(-1.0, 1.0, w, device=device, dtype=dtype)
            ys = torch.linspace(-1.0, 1.0, h, device=device, dtype=dtype)
        else:
            xs = torch.arange(w, device=device, dtype=dtype)
            ys = torch.arange(h, device=device, dtype=dtype)

        yy, xx = torch.meshgrid(ys, xs, indexing="ij")
        coord = torch.stack([xx, yy], dim=0).unsqueeze(0)  # [1,2,H,W]
        return coord

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, _, h, w = x.shape
        coord = self._make_coord_grid(h, w, x.device, x.dtype, self.normalize)
        if coord.shape[0] != b:
            coord = coord.repeat(b, 1, 1, 1)  # [B,2,H,W]
        x = torch.cat([x, coord], dim=1)      # [B,C+2,H,W]
        return self.conv(x)


# =========================================================
# SobelEdgePool (shape-preserving)
# =========================================================
class SobelEdgePool(nn.Module):
    """
    Sobel edge magnitude extracted per-channel (depthwise).
    Output shape == input shape. No learnable params.
    """
    def __init__(self, eps: float = 1e-6):
        super().__init__()
        self.eps = eps

        kx = torch.tensor(
            [[-1, 0, 1],
             [-2, 0, 2],
             [-1, 0, 1]], dtype=torch.float32
        )
        ky = torch.tensor(
            [[-1, -2, -1],
             [ 0,  0,  0],
             [ 1,  2,  1]], dtype=torch.float32
        )

        # register as buffers; will be expanded to [C,1,3,3] at runtime
        self.register_buffer("kx", kx.view(1, 1, 3, 3), persistent=False)
        self.register_buffer("ky", ky.view(1, 1, 3, 3), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        kx = self.kx.to(dtype=x.dtype, device=x.device).expand(c, 1, 3, 3)
        ky = self.ky.to(dtype=x.dtype, device=x.device).expand(c, 1, 3, 3)

        # depthwise conv per-channel
        gx = F.conv2d(x, kx, bias=None, stride=1, padding=1, groups=c)
        gy = F.conv2d(x, ky, bias=None, stride=1, padding=1, groups=c)

        # magnitude (stable)
        mag = torch.sqrt(gx * gx + gy * gy + self.eps)
        return mag


# =========================================================
# ECA (Efficient Channel Attention)
# =========================================================
class ECALayer(nn.Module):
    """
    Efficient Channel Attention:
      GAP -> 1D conv over channels -> sigmoid -> scale
    Very light.
    """
    def __init__(self, channels: int, k_size: int = 3):
        super().__init__()
        k_size = int(k_size)
        if k_size % 2 == 0:
            k_size += 1  # ensure odd
        self.conv1d = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B,C,H,W]
        y = x.mean(dim=(2, 3), keepdim=False)       # [B,C]
        y = y.unsqueeze(1)                         # [B,1,C]
        y = self.conv1d(y)                         # [B,1,C]
        y = self.sigmoid(y).squeeze(1).unsqueeze(-1).unsqueeze(-1)  # [B,C,1,1]
        return x * y


# =========================================================
# DepthMix (light refinement / compression-friendly)
# =========================================================
class DepthMix(nn.Module):
    """
    Simple 1x1 Conv + BN + SiLU (shape-preserving).
    Used as lightweight refinement (paper uses it to balance compute).
    """
    def __init__(self, c: int):
        super().__init__()
        self.conv = nn.Conv2d(c, c, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn = nn.BatchNorm2d(c)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.bn(self.conv(x)))


# =========================================================
# MAPBoost (full block with SobelEdgePool)
# =========================================================
class MAPBoost(nn.Module):
    """
    MAPBoost (full) block (paper-inspired), shape-preserving by default (s=1).
    Pipeline:
      CoordConv2d -> BN -> SiLU
      -> DepthwiseConv3x3 -> BN -> SiLU
      -> (x + alpha * SobelEdgePool(x))    # residual edge injection
      -> ECA
      -> DepthMix (1x1 + BN + SiLU)
      -> 1x1 Conv(+BN+SiLU) to c2
      -> optional residual if c1==c2 and s==1

    Notes:
    - For head/neck replacement, typically use s=1.
    - If you replace a block that changes channels (c1!=c2), residual skip is disabled automatically.
    """
    def __init__(
        self,
        c1: int,
        c2: int,
        k_coord: int = 3,
        s: int = 1,
        normalize: bool = True,
        eca_k: int = 3,
    ):
        super().__init__()
        self.c1, self.c2, self.s = c1, c2, s

        # CoordConv stage
        self.coord = CoordConv2d(c1, c2, k=k_coord, s=s, p=None, g=1, d=1, bias=False, normalize=normalize)
        self.bn0 = nn.BatchNorm2d(c2)
        self.act0 = nn.SiLU(inplace=True)

        # Depthwise conv stage (compute rebalance)
        self.dw = nn.Conv2d(c2, c2, kernel_size=3, stride=1, padding=1, groups=c2, bias=False)
        self.bn1 = nn.BatchNorm2d(c2)
        self.act1 = nn.SiLU(inplace=True)

        # Sobel edge injection
        self.sobel = SobelEdgePool()
        # learnable scale for edge to avoid destabilizing early training
        self.alpha = nn.Parameter(torch.tensor(0.1, dtype=torch.float32))

        # Channel attention
        self.eca = ECALayer(c2, k_size=eca_k)

        # DepthMix refinement
        self.mix = DepthMix(c2)

        # Final 1x1 projection (keeps c2, but stabilizes)
        self.proj = nn.Conv2d(c2, c2, kernel_size=1, stride=1, padding=0, bias=False)
        self.bn2 = nn.BatchNorm2d(c2)
        self.act2 = nn.SiLU(inplace=True)

        # Residual possible only if shape matches and stride==1
        self.use_skip = (c1 == c2) and (s == 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x

        x = self.act0(self.bn0(self.coord(x)))
        x = self.act1(self.bn1(self.dw(x)))

        # Edge injection (shape-preserving)
        edge = self.sobel(x)
        x = x + self.alpha.to(dtype=x.dtype, device=x.device) * edge

        x = self.eca(x)
        x = self.mix(x)

        x = self.act2(self.bn2(self.proj(x)))

        if self.use_skip:
            x = x + identity
        return x


Writing hoangtn_MAPBoost.py


In [17]:
# train_loop.py
import argparse
from dataclasses import dataclass

import pandas as pd
import torch

############################################################
# Custom modules
from hoangtn_MAPBoost import MAPBoost
from torch import Tensor
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset

from ultralytics import YOLO

############################################################


# -----------------------------
# Utils
# -----------------------------
def get_device() -> torch.device:
    # Force cuda:0 on Kaggle when available
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


# -----------------------------
# Dataset
# -----------------------------
class AodDataset(Dataset):
    """
    Read images + YOLO labels (cls xc yc w h) normalized [0..1].

    Return:
      - image: uint8 tensor [C,H,W] on CPU
      - targets: float tensor [N,5] on CPU (or [[-1]*5] if empty/missing).
    """

    def __init__(
        self,
        images_dir: str,
        labels_dir: str,
        images_ext: str = "jpg",
        labels_ext: str = "txt",
        transforms=None,
        verbose_bad_labels: bool = False,
    ):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.images_ext = images_ext
        self.labels_ext = labels_ext
        self.transforms = transforms
        self.verbose_bad_labels = verbose_bad_labels

        img_glob = os.path.join(images_dir, f"*.{images_ext}")
        self.images = sorted(glob.glob(img_glob))
        if len(self.images) == 0:
            raise FileNotFoundError(f"No images found: {img_glob}")

        self.miss_labeled: list[str] = []

    def _labels_path_from_image(self, image_path: str) -> str:
        base = os.path.basename(image_path)
        stem = base[: -(len(self.images_ext) + 1)]
        return os.path.join(self.labels_dir, f"{stem}.{self.labels_ext}")

    def _read_labels(self, labels_path: str) -> Tensor:
        if not os.path.exists(labels_path):
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        labels: list[list[float]] = []
        with open(labels_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) != 5:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid values count: {line}")
                    continue
                try:
                    c, xc, yc, w, h = map(float, parts)
                except ValueError:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: cannot parse: {line}")
                    continue

                if not (0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid bbox: {line}")
                    continue

                labels.append([c, xc, yc, w, h])

        if len(labels) == 0:
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        return torch.tensor(labels, dtype=torch.float32)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        image_path = self.images[idx]
        img = decode_image(image_path)  # uint8 [C,H,W] on CPU
        if self.transforms is not None:
            img = self.transforms(img)

        labels_path = self._labels_path_from_image(image_path)
        targets = self._read_labels(labels_path)
        return img, targets

    def __len__(self) -> int:
        return len(self.images)


# -----------------------------
# Collate & Batch builder
# -----------------------------
def collate_fn(batch: list[tuple[Tensor, Tensor]]) -> tuple[Tensor, Tensor]:
    images, targets = zip(*batch)
    images = torch.stack(images, dim=0)
    padded_targets = pad_sequence(list(targets), batch_first=True, padding_value=-1.0)
    return images, padded_targets


def prepare_batch_dict(images: Tensor, targets: Tensor) -> dict:
    valid_mask = targets[..., 0] != -1
    if not valid_mask.any():
        return {
            "img": images,
            "batch_idx": torch.empty((0,), device=images.device, dtype=torch.int64),
            "cls": torch.empty((0,), device=images.device, dtype=torch.float32),
            "bboxes": torch.empty((0, 4), device=images.device, dtype=torch.float32),
        }

    valid_targets = targets[valid_mask]  # [M,5]

    per_img_counts = valid_mask.sum(dim=1).to(torch.int64)
    batch_idx_list = []
    for i, cnt in enumerate(per_img_counts.tolist()):
        if cnt > 0:
            batch_idx_list.append(torch.full((cnt,), i, device=images.device, dtype=torch.int64))
    batch_idx = (
        torch.cat(batch_idx_list, dim=0)
        if batch_idx_list
        else torch.empty((0,), device=images.device, dtype=torch.int64)
    )

    cls = valid_targets[:, 0].to(torch.float32)
    bboxes = valid_targets[:, 1:].to(torch.float32)
    return {"img": images, "batch_idx": batch_idx, "cls": cls, "bboxes": bboxes}


# -----------------------------
# Train / Valid
# -----------------------------
def train_one_epoch(
    model: YOLO,
    optimizer: Adam,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_train_steps: int = 0,
    debug_print_first_batch: bool = True,
) -> pd.DataFrame:
    model.model.train(True)
    torch.set_grad_enabled(True)
    loss_fn = v8DetectionLoss(model.model)

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"train epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_train_steps > 0 and step >= max_train_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        if debug_print_first_batch and step == 0:
            print("[DEBUG] images device:", images.device)
            print("[DEBUG] targets device:", targets.device)
            print("[DEBUG] model device:", next(model.model.parameters()).device)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            pbar.set_postfix_str("skip(empty)")
            continue

        optimizer.zero_grad(set_to_none=True)

        # Prevent inference_mode leakage from Ultralytics internals in notebooks
        with torch.inference_mode(False):
            with torch.enable_grad():
                preds = model.model(images)
                batch_loss, last_loss = loss_fn(preds, batch)
                loss_scalar = batch_loss.sum()

        if not torch.isfinite(loss_scalar):
            pbar.set_postfix_str("skip(non-finite)")
            continue

        if not loss_scalar.requires_grad:
            raise RuntimeError(
                f"loss_scalar.requires_grad=False at step {step}. "
                f"Grad is disabled globally or inference_mode is active."
            )

        loss_scalar.backward()
        optimizer.step()

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        row = {
            "epoch": epoch,
            "step": step,
            "box": float(box_loss),
            "cls": float(cls_loss),
            "dfl": float(dfl_loss),
        }
        rows.append(row)
        pbar.set_postfix({"box": f"{row['box']:.4f}", "cls": f"{row['cls']:.4f}", "dfl": f"{row['dfl']:.4f}"})

    return pd.DataFrame(rows)


@torch.no_grad()
def valid_loss_only(
    model: YOLO,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_val_steps: int = 0,
) -> pd.DataFrame:
    model.model.eval()
    loss_fn = v8DetectionLoss(model.model)

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"valid(loss) epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_val_steps > 0 and step >= max_val_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            continue

        preds = model.model(images)
        _, last_loss = loss_fn(preds, batch)

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        rows.append(
            {"epoch": epoch, "batch": step, "box": float(box_loss), "cls": float(cls_loss), "dfl": float(dfl_loss)}
        )

    return pd.DataFrame(rows)


def yolo_val_metrics(model: YOLO, data_yaml: str, device_id=0, workers: int = 0):
    return model.val(data=data_yaml, device=device_id, workers=workers, verbose=False, plots=False)


# -----------------------------
# Main
# -----------------------------
@dataclass
class Config:
    base_path: str
    data_yaml: str
    epochs: int = 2
    batch_size: int = 4
    lr: float = 3e-4
    num_workers: int = 2
    img_ext: str = "jpg"
    label_ext: str = "txt"
    checkpoint_dir: str = "checkpoints"
    max_train_steps: int = 0
    max_val_steps: int = 0


def main(cfg: Config):
    device = get_device()
    print(f"[INFO] device = {device}")
    if device.type == "cuda":
        print("[INFO] CUDA available:", torch.cuda.is_available())
        print("[INFO] CUDA device:", torch.cuda.get_device_name(0))

    ensure_dir(cfg.checkpoint_dir)

    # 1) load YOLO wrapper
    model = YOLO(model="custom_yolov11n.yaml", task="detect", verbose=True)

    # 2) set loss weights
    model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore

    # 3) SWAP layers
    layers = model.model.model  # nn.ModuleList

    def is_ultra_conv(m: nn.Module) -> bool:
        # Ultralytics Conv wrapper usually has .conv (nn.Conv2d)
        return hasattr(m, "conv") and isinstance(getattr(m, "conv", None), nn.Conv2d)

    def get_conv_meta(m: nn.Module):
        # (in_ch, out_ch, stride) for Ultralytics Conv wrapper
        if is_ultra_conv(m):
            conv = m.conv
            s = conv.stride[0] if isinstance(conv.stride, tuple) else int(conv.stride)
            return conv.in_channels, conv.out_channels, s
        return None

    def get_c3k2_meta(m: nn.Module):
        # (c1,c2) for C3k2-like modules (Ultralytics)
        if hasattr(m, "cv1") and hasattr(m, "cv2") and hasattr(m.cv1, "conv") and hasattr(m.cv2, "conv"):
            try:
                return m.cv1.conv.in_channels, m.cv2.conv.out_channels
            except Exception:
                return None
        return None

    # -------------------------
    # A) Swap MAPBoostLite into backbone downsample convs
    # -------------------------
    targets = {"p2": None, "p3": None}

    # Find Conv stride=2 with out=128 (P2/4) and out=256 (P3/8)
    for i, m in enumerate(layers):
        meta = get_conv_meta(m)
        if meta is None:
            continue
        c1, c2, s = meta
        if s == 2 and c2 == 128 and targets["p2"] is None:
            targets["p2"] = i
        if s == 2 and c2 == 256 and targets["p3"] is None:
            targets["p3"] = i

    if targets["p2"] is None or targets["p3"] is None:
        print("\n[DEBUG] List Conv(stride=2):")
        for i, m in enumerate(layers):
            meta = get_conv_meta(m)
            if meta is None:
                continue
            c1, c2, s = meta
            if s == 2:
                print(f"  idx={i:3d}  Conv  {c1}->{c2}  stride={s}")
        raise RuntimeError(f"Không tìm thấy Conv stride=2 cho P2/4(out=128) hoặc P3/8(out=256). Found={targets}")

    # Swap P2/4
    idx_p2 = targets["p2"]
    old_p2 = layers[idx_p2]
    c1_p2, c2_p2, _s_p2 = get_conv_meta(old_p2)
    new_p2 = MAPBoostLite(c1_p2, c2_p2, k=3, s=2, normalize=True)

    for k in ("f", "i", "type"):  # DO NOT copy "np"
        if hasattr(old_p2, k):
            setattr(new_p2, k, getattr(old_p2, k))
    new_p2.training = old_p2.training
    layers[idx_p2] = new_p2
    print(f"[SWAP OK] P2/4 at idx={idx_p2}: Conv({c1_p2}->{c2_p2}, s=2) -> MAPBoostLite")

    # Swap P3/8
    idx_p3 = targets["p3"]
    old_p3 = layers[idx_p3]
    c1_p3, c2_p3, _s_p3 = get_conv_meta(old_p3)
    new_p3 = MAPBoostLite(c1_p3, c2_p3, k=3, s=2, normalize=True)

    for k in ("f", "i", "type"):  # DO NOT copy "np"
        if hasattr(old_p3, k):
            setattr(new_p3, k, getattr(old_p3, k))
    new_p3.training = old_p3.training
    layers[idx_p3] = new_p3
    print(f"[SWAP OK] P3/8 at idx={idx_p3}: Conv({c1_p3}->{c2_p3}, s=2) -> MAPBoostLite")

    # -------------------------
    # B) Swap MAPBoost(full) into head C3k2 blocks (P3/P4, optionally P5)
    # -------------------------
    targets_head = {"p3": None, "p4": None, "p5": None}

    for i, m in enumerate(layers):
        meta = get_c3k2_meta(m)
        if meta is None:
            continue
        c1, c2 = meta

        # focus head/neck area (after backbone), as in your printed summary
        if i <= 10:
            continue

        # match the known head blocks from your log
        if c1 == 256 and c2 == 64 and targets_head["p3"] is None:
            targets_head["p3"] = i
        elif c1 == 192 and c2 == 128 and targets_head["p4"] is None:
            targets_head["p4"] = i
        elif c1 == 384 and c2 == 256 and targets_head["p5"] is None:
            targets_head["p5"] = i

    print("[INFO] head C3k2 targets:", targets_head)

    enable_p5 = False  # set True if you want to replace P5 head too
    swap_mapboost_idxs = []

    def swap_to_mapboost(idx: int):
        old = layers[idx]
        c1, c2 = get_c3k2_meta(old)
        new = MAPBoost(c1, c2, s=1)  # head is shape-preserving (s=1)

        for k in ("f", "i", "type"):  # DO NOT copy "np"
            if hasattr(old, k):
                setattr(new, k, getattr(old, k))
        new.training = old.training
        layers[idx] = new
        swap_mapboost_idxs.append(idx)
        print(f"[SWAP OK] HEAD idx={idx}: C3k2({c1}->{c2}) -> MAPBoost(full)")

    if targets_head["p3"] is None or targets_head["p4"] is None:
        raise RuntimeError(f"Không tìm thấy C3k2 head P3/P4 theo meta 256->64 và 192->128. Found={targets_head}")

    swap_to_mapboost(targets_head["p3"])
    swap_to_mapboost(targets_head["p4"])

    if enable_p5:
        if targets_head["p5"] is None:
            raise RuntimeError(
                f"enable_p5=True nhưng không tìm thấy C3k2 head P5 theo meta 384->256. Found={targets_head}"
            )
        swap_to_mapboost(targets_head["p5"])

    # 4) move to device (after swaps)
    model.model.to(device)  # type: ignore

    # -------------------------
    # Print model info
    # -------------------------
    print("\n================ MODEL INFO (after swap) ================\n", flush=True)
    model.model.info()

    print("\n================ LAYERS (after swap) ================\n", flush=True)

    swapped_idxs = {idx_p2, idx_p3, *swap_mapboost_idxs}

    for i, m in enumerate(model.model.model):
        name = m.__class__.__name__
        f = getattr(m, "f", None)
        np_ = sum(p.numel() for p in m.parameters()) if isinstance(m, nn.Module) else 0

        c1 = c2 = None
        if hasattr(m, "cv1") and hasattr(m, "cv2") and hasattr(m.cv1, "conv") and hasattr(m.cv2, "conv"):
            try:
                c1 = m.cv1.conv.in_channels
                c2 = m.cv2.conv.out_channels
            except Exception:
                pass

        mark = "  <== SWAPPED" if i in swapped_idxs else ""

        if c1 is not None and c2 is not None:
            print(f"{i:3d}  f={f!s:>6}  {name:<20}  c1->c2: {c1:4d}->{c2:<4d}  params={np_:>8d}{mark}")
        else:
            print(f"{i:3d}  f={f!s:>6}  {name:<20}  params={np_:>8d}{mark}")

    print("[DEBUG] model param device:", next(model.model.parameters()).device)

    # -------------------------
    # Optimizer + Data
    # -------------------------
    optimizer = Adam(model.model.parameters(), lr=cfg.lr)

    train_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "train/images"),
        labels_dir=os.path.join(cfg.base_path, "train/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )
    val_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "valid/images"),
        labels_dir=os.path.join(cfg.base_path, "valid/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )

    pin = torch.cuda.is_available()
    workers = cfg.num_workers

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )

    # -------------------------
    # Train
    # -------------------------
    train_all = []

    best_score = float("inf")
    best_epoch = -1
    best_path = os.path.join(cfg.checkpoint_dir, "best.pt")
    last_path = os.path.join(cfg.checkpoint_dir, "last.pt")

    for epoch in range(1, cfg.epochs + 1):
        print(f"\n==================== EPOCH {epoch}/{cfg.epochs} ====================")

        train_df = train_one_epoch(
            model,
            optimizer,
            train_loader,
            device,
            epoch,
            max_train_steps=cfg.max_train_steps,
            debug_print_first_batch=(epoch == 1),
        )
        train_all.append(train_df)

        if len(train_df) > 0:
            train_score = train_df[["box", "cls", "dfl"]].mean().sum()
            print(f"[EPOCH {epoch}] train_score={train_score:.6f} | best_score={best_score:.6f}")

            if train_score < best_score:
                best_score = train_score
                best_epoch = epoch
                model.save(best_path)
                print(f"[BEST] updated → epoch {best_epoch}, saved to {best_path}")
        else:
            print("[TRAIN] no valid batches produced loss (all empty labels?)")

        model.save(last_path)
        ckpt_path = os.path.join(cfg.checkpoint_dir, f"epoch_{epoch}.pt")
        model.save(ckpt_path)
        print(f"[INFO] saved checkpoint: {ckpt_path} (and last.pt)")

    train_csv = os.path.join(cfg.checkpoint_dir, "train_losses.csv")
    pd.concat(train_all, axis=0, ignore_index=True).to_csv(train_csv, index=False)
    print(f"[INFO] saved train logs: {train_csv}")

    print("\n==================== TRAIN SUMMARY ====================")
    print(f"[BEST MODEL] epoch = {best_epoch}")
    print(f"[BEST MODEL] score = {best_score:.6f}")
    print(f"[BEST MODEL] path  = {best_path}")
    print(f"[LAST MODEL] path  = {last_path}")

    # -------------------------
    # Final evaluation
    # -------------------------
    print("\n==================== FINAL EVALUATION ====================")

    eval_ckpt = best_path if os.path.exists(best_path) else last_path
    print(f"[EVAL] loading checkpoint: {eval_ckpt}")

    eval_model = YOLO(eval_ckpt)
    eval_model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore
    eval_model.model.to(device)  # type: ignore

    val_loss_df = valid_loss_only(
        eval_model,
        val_loader,
        device,
        epoch=best_epoch if os.path.exists(best_path) else cfg.epochs,
        max_val_steps=cfg.max_val_steps,
    )
    val_csv = os.path.join(cfg.checkpoint_dir, "val_losses.csv")
    val_loss_df.to_csv(val_csv, index=False)
    if len(val_loss_df) > 0:
        print("[FINAL VALID-LOSS] mean:", val_loss_df[["box", "cls", "dfl"]].mean().to_dict())
    else:
        print("[FINAL VALID-LOSS] no valid batches (empty labels?)")
    print(f"[INFO] saved val losses: {val_csv}")

    try:
        print("[FINAL VALID-METRICS] running ultralytics val() ...")
        device_id = 0 if device.type == "cuda" else "cpu"
        metrics = yolo_val_metrics(eval_model, cfg.data_yaml, device_id=device_id, workers=cfg.num_workers)

        print(f"Precision: {metrics.box.mp:.4f}")
        print(f"Recall:    {metrics.box.mr:.4f}")
        print(f"mAP@0.5:   {metrics.box.map50:.4f}")
        print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
    except Exception as e:
        print(f"[WARN] final model.val() failed: {e}")

    return model


if __name__ == "__main__":

    def running_in_notebook():
        try:
            from IPython import get_ipython

            return get_ipython() is not None
        except Exception:
            return False

    if running_in_notebook():
        print("[INFO] Running in Kaggle/Colab Notebook → using hardcoded config")
        cfg = Config(
            base_path="/kaggle/input/aod-4-yolo",
            data_yaml="/kaggle/working/data.yaml",
            epochs=4,
            batch_size=4,
            lr=3e-4,
            num_workers=2,
            max_train_steps=1000,
            max_val_steps=0,
        )
        main(cfg)
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument("--base_path", type=str, required=True)
        parser.add_argument("--data_yaml", type=str, required=True)
        parser.add_argument("--epochs", type=int, default=2)
        parser.add_argument("--batch_size", type=int, default=4)
        parser.add_argument("--lr", type=float, default=3e-4)
        parser.add_argument("--num_workers", type=int, default=2)
        parser.add_argument("--img_ext", type=str, default="jpg")
        parser.add_argument("--label_ext", type=str, default="txt")
        parser.add_argument("--checkpoint_dir", type=str, default="checkpoints")
        parser.add_argument("--max_train_steps", type=int, default=0)
        parser.add_argument("--max_val_steps", type=int, default=0)

        args = parser.parse_args()
        cfg = Config(**vars(args))
        main(cfg)

[INFO] Running in Kaggle/Colab Notebook → using hardcoded config
[INFO] device = cuda:0
[INFO] CUDA available: True
[INFO] CUDA device: Tesla P100-PCIE-16GB

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultral

train epoch 1:   0%|          | 0/3941 [00:00<?, ?it/s]

[DEBUG] images device: cuda:0
[DEBUG] targets device: cuda:0
[DEBUG] model device: cuda:0
[EPOCH 1] train_score=15.108537 | best_score=inf
[BEST] updated → epoch 1, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_1.pt (and last.pt)

==================== EPOCH 2/4 ====================


train epoch 2:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 2] train_score=11.417528 | best_score=15.108537
[BEST] updated → epoch 2, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_2.pt (and last.pt)

==================== EPOCH 3/4 ====================


...

[EPOCH 3] train_score=10.311686 | best_score=11.417528
[BEST] updated → epoch 3, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_3.pt (and last.pt)

==================== EPOCH 4/4 ====================


train epoch 4:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 4] train_score=9.636256 | best_score=10.311686
[BEST] updated → epoch 4, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_4.pt (and last.pt)
[INFO] saved train logs: checkpoints/train_losses.csv

==================== TRAIN SUMMARY ====================
[BEST MODEL] epoch = 4
[BEST MODEL] score = 9.636256
[BEST MODEL] path  = checkpoints/best.pt
[LAST MODEL] path  = checkpoints/last.pt

==================== FINAL EVALUATION ====================
[EVAL] loading checkpoint: checkpoints/best.pt


valid(loss) epoch 4:   0%|          | 0/1129 [00:00<?, ?it/s]

[FINAL VALID-LOSS] mean: {'box': 4.204161286354065, 'cls': 2.5325559619830864, 'dfl': 3.6265973657770543}
[INFO] saved val losses: checkpoints/val_losses.csv
[FINAL VALID-METRICS] running ultralytics val() ...
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
custom_YOLOv11n summary: 127 layers, 2,888,268 parameters, 879,636 gradients, 8.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 38.0±8.9 MB/s, size: 41.2 KB)
val: Scanning /kaggle/input/aod-4-yolo/valid/labels... 4514 images, 125 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4514/4514 893.2it/s 5.1s<0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/aod-4-yolo/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 283/283 10.4it/s 27.2s<0.1s
                   all       4514       6369      0.291       0.26      0.204     0.0711
Speed: 0.8ms preprocess, 2.8ms inference, 0.0ms loss, 0.9ms po

#  Bước 2 – Bước B (tiếp theo):

# Thêm WeightedConcat để thay Concat trước head refine
# (nhưng vẫn giữ Detect 3 head)


In [18]:
%%writefile hoangtn_WeightedConcat.py

import torch
import torch.nn as nn
from typing import List


class WeightedConcat(nn.Module):
    """
    WeightedConcat
    - Drop-in replacement for Ultralytics Concat
    - Learns a weight for each input feature map
    - Applies softmax-normalized weights before concatenation

    Forward:
        inputs: List[Tensor] with same H,W
        output: torch.cat(weighted_inputs, dim=channel)
    """

    def __init__(self, dimension: int = 1, n_inputs: int = 2, eps: float = 1e-4):
        """
        Args:
            dimension: concat dimension (YOLO uses channel dim = 1)
            n_inputs: number of input feature maps
            eps: numerical stability
        """
        super().__init__()
        self.d = dimension
        self.n_inputs = n_inputs
        self.eps = eps

        # Learnable weights (initialized equal)
        self.w = nn.Parameter(torch.ones(n_inputs), requires_grad=True)

    def forward(self, inputs: List[torch.Tensor]) -> torch.Tensor:
        assert isinstance(inputs, (list, tuple)), "WeightedConcat expects list/tuple of tensors"
        assert len(inputs) == self.n_inputs, (
            f"Expected {self.n_inputs} inputs, got {len(inputs)}"
        )

        # Softmax-normalized weights
        weights = torch.softmax(self.w, dim=0)

        # Apply weights
        weighted = []
        for i, x in enumerate(inputs):
            weighted.append(x * weights[i])

        # Concatenate
        return torch.cat(weighted, dim=self.d)


Writing hoangtn_WeightedConcat.py


In [19]:
# train_loop.py
import argparse
from dataclasses import dataclass

import pandas as pd
import torch

############################################################
# Custom modules
from hoangtn_WeightedConcat import WeightedConcat
from torch import Tensor
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset

from ultralytics import YOLO

############################################################


# -----------------------------
# Utils
# -----------------------------
def get_device() -> torch.device:
    # Force cuda:0 on Kaggle when available
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


# -----------------------------
# Dataset
# -----------------------------
class AodDataset(Dataset):
    """
    Read images + YOLO labels (cls xc yc w h) normalized [0..1].

    Return:
      - image: uint8 tensor [C,H,W] on CPU
      - targets: float tensor [N,5] on CPU (or [[-1]*5] if empty/missing).
    """

    def __init__(
        self,
        images_dir: str,
        labels_dir: str,
        images_ext: str = "jpg",
        labels_ext: str = "txt",
        transforms=None,
        verbose_bad_labels: bool = False,
    ):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.images_ext = images_ext
        self.labels_ext = labels_ext
        self.transforms = transforms
        self.verbose_bad_labels = verbose_bad_labels

        img_glob = os.path.join(images_dir, f"*.{images_ext}")
        self.images = sorted(glob.glob(img_glob))
        if len(self.images) == 0:
            raise FileNotFoundError(f"No images found: {img_glob}")

        self.miss_labeled: list[str] = []

    def _labels_path_from_image(self, image_path: str) -> str:
        base = os.path.basename(image_path)
        stem = base[: -(len(self.images_ext) + 1)]
        return os.path.join(self.labels_dir, f"{stem}.{self.labels_ext}")

    def _read_labels(self, labels_path: str) -> Tensor:
        if not os.path.exists(labels_path):
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        labels: list[list[float]] = []
        with open(labels_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) != 5:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid values count: {line}")
                    continue
                try:
                    c, xc, yc, w, h = map(float, parts)
                except ValueError:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: cannot parse: {line}")
                    continue

                if not (0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid bbox: {line}")
                    continue

                labels.append([c, xc, yc, w, h])

        if len(labels) == 0:
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        return torch.tensor(labels, dtype=torch.float32)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        image_path = self.images[idx]
        img = decode_image(image_path)  # uint8 [C,H,W] on CPU
        if self.transforms is not None:
            img = self.transforms(img)

        labels_path = self._labels_path_from_image(image_path)
        targets = self._read_labels(labels_path)
        return img, targets

    def __len__(self) -> int:
        return len(self.images)


# -----------------------------
# Collate & Batch builder
# -----------------------------
def collate_fn(batch: list[tuple[Tensor, Tensor]]) -> tuple[Tensor, Tensor]:
    images, targets = zip(*batch)
    images = torch.stack(images, dim=0)
    padded_targets = pad_sequence(list(targets), batch_first=True, padding_value=-1.0)
    return images, padded_targets


def prepare_batch_dict(images: Tensor, targets: Tensor) -> dict:
    valid_mask = targets[..., 0] != -1
    if not valid_mask.any():
        return {
            "img": images,
            "batch_idx": torch.empty((0,), device=images.device, dtype=torch.int64),
            "cls": torch.empty((0,), device=images.device, dtype=torch.float32),
            "bboxes": torch.empty((0, 4), device=images.device, dtype=torch.float32),
        }

    valid_targets = targets[valid_mask]  # [M,5]

    per_img_counts = valid_mask.sum(dim=1).to(torch.int64)
    batch_idx_list = []
    for i, cnt in enumerate(per_img_counts.tolist()):
        if cnt > 0:
            batch_idx_list.append(torch.full((cnt,), i, device=images.device, dtype=torch.int64))
    batch_idx = (
        torch.cat(batch_idx_list, dim=0)
        if batch_idx_list
        else torch.empty((0,), device=images.device, dtype=torch.int64)
    )

    cls = valid_targets[:, 0].to(torch.float32)
    bboxes = valid_targets[:, 1:].to(torch.float32)
    return {"img": images, "batch_idx": batch_idx, "cls": cls, "bboxes": bboxes}


# -----------------------------
# Train / Valid
# -----------------------------
def train_one_epoch(
    model: YOLO,
    optimizer: Adam,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_train_steps: int = 0,
    debug_print_first_batch: bool = True,
) -> pd.DataFrame:
    model.model.train(True)
    torch.set_grad_enabled(True)
    loss_fn = v8DetectionLoss(model.model)

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"train epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_train_steps > 0 and step >= max_train_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        if debug_print_first_batch and step == 0:
            print("[DEBUG] images device:", images.device)
            print("[DEBUG] targets device:", targets.device)
            print("[DEBUG] model device:", next(model.model.parameters()).device)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            pbar.set_postfix_str("skip(empty)")
            continue

        optimizer.zero_grad(set_to_none=True)

        with torch.inference_mode(False):
            with torch.enable_grad():
                preds = model.model(images)
                batch_loss, last_loss = loss_fn(preds, batch)
                loss_scalar = batch_loss.sum()

        if not torch.isfinite(loss_scalar):
            pbar.set_postfix_str("skip(non-finite)")
            continue

        if not loss_scalar.requires_grad:
            raise RuntimeError(
                f"loss_scalar.requires_grad=False at step {step}. "
                f"Grad is disabled globally or inference_mode is active."
            )

        loss_scalar.backward()
        optimizer.step()

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        row = {"epoch": epoch, "step": step, "box": float(box_loss), "cls": float(cls_loss), "dfl": float(dfl_loss)}
        rows.append(row)
        pbar.set_postfix({"box": f"{row['box']:.4f}", "cls": f"{row['cls']:.4f}", "dfl": f"{row['dfl']:.4f}"})

    return pd.DataFrame(rows)


@torch.no_grad()
def valid_loss_only(
    model: YOLO,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_val_steps: int = 0,
) -> pd.DataFrame:
    model.model.eval()
    loss_fn = v8DetectionLoss(model.model)

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"valid(loss) epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_val_steps > 0 and step >= max_val_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            continue

        preds = model.model(images)
        _, last_loss = loss_fn(preds, batch)

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        rows.append(
            {"epoch": epoch, "batch": step, "box": float(box_loss), "cls": float(cls_loss), "dfl": float(dfl_loss)}
        )

    return pd.DataFrame(rows)


def yolo_val_metrics(model: YOLO, data_yaml: str, device_id=0, workers: int = 0):
    return model.val(data=data_yaml, device=device_id, workers=workers, verbose=False, plots=False)


# -----------------------------
# Main
# -----------------------------
@dataclass
class Config:
    base_path: str
    data_yaml: str
    epochs: int = 2
    batch_size: int = 4
    lr: float = 3e-4
    num_workers: int = 2
    img_ext: str = "jpg"
    label_ext: str = "txt"
    checkpoint_dir: str = "checkpoints"
    max_train_steps: int = 0
    max_val_steps: int = 0


def main(cfg: Config):
    device = get_device()
    print(f"[INFO] device = {device}")
    if device.type == "cuda":
        print("[INFO] CUDA available:", torch.cuda.is_available())
        print("[INFO] CUDA device:", torch.cuda.get_device_name(0))

    ensure_dir(cfg.checkpoint_dir)

    # 1) load YOLO wrapper
    model = YOLO(model="custom_yolov11n.yaml", task="detect", verbose=True)

    # 2) set loss weights
    model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore

    # 3) SWAP layers
    layers = model.model.model  # nn.ModuleList

    def is_ultra_conv(m: nn.Module) -> bool:
        return hasattr(m, "conv") and isinstance(getattr(m, "conv", None), nn.Conv2d)

    def get_conv_meta(m: nn.Module):
        if is_ultra_conv(m):
            conv = m.conv
            s = conv.stride[0] if isinstance(conv.stride, tuple) else int(conv.stride)
            return conv.in_channels, conv.out_channels, s
        return None

    def get_c3k2_meta(m: nn.Module):
        if hasattr(m, "cv1") and hasattr(m, "cv2") and hasattr(m.cv1, "conv") and hasattr(m.cv2, "conv"):
            try:
                return m.cv1.conv.in_channels, m.cv2.conv.out_channels
            except Exception:
                return None
        return None

    # -------------------------
    # A) Swap MAPBoostLite into backbone downsample convs
    # -------------------------
    targets = {"p2": None, "p3": None}

    for i, m in enumerate(layers):
        meta = get_conv_meta(m)
        if meta is None:
            continue
        c1, c2, s = meta
        if s == 2 and c2 == 128 and targets["p2"] is None:
            targets["p2"] = i
        if s == 2 and c2 == 256 and targets["p3"] is None:
            targets["p3"] = i

    if targets["p2"] is None or targets["p3"] is None:
        print("\n[DEBUG] List Conv(stride=2):")
        for i, m in enumerate(layers):
            meta = get_conv_meta(m)
            if meta is None:
                continue
            c1, c2, s = meta
            if s == 2:
                print(f"  idx={i:3d}  Conv  {c1}->{c2}  stride={s}")
        raise RuntimeError(f"Không tìm thấy Conv stride=2 cho P2/4(out=128) hoặc P3/8(out=256). Found={targets}")

    idx_p2 = targets["p2"]
    old_p2 = layers[idx_p2]
    c1_p2, c2_p2, _s_p2 = get_conv_meta(old_p2)
    new_p2 = MAPBoostLite(c1_p2, c2_p2, k=3, s=2, normalize=True)

    for k in ("f", "i", "type"):
        if hasattr(old_p2, k):
            setattr(new_p2, k, getattr(old_p2, k))
    new_p2.training = old_p2.training
    layers[idx_p2] = new_p2
    print(f"[SWAP OK] P2/4 at idx={idx_p2}: Conv({c1_p2}->{c2_p2}, s=2) -> MAPBoostLite")

    idx_p3 = targets["p3"]
    old_p3 = layers[idx_p3]
    c1_p3, c2_p3, _s_p3 = get_conv_meta(old_p3)
    new_p3 = MAPBoostLite(c1_p3, c2_p3, k=3, s=2, normalize=True)

    for k in ("f", "i", "type"):
        if hasattr(old_p3, k):
            setattr(new_p3, k, getattr(old_p3, k))
    new_p3.training = old_p3.training
    layers[idx_p3] = new_p3
    print(f"[SWAP OK] P3/8 at idx={idx_p3}: Conv({c1_p3}->{c2_p3}, s=2) -> MAPBoostLite")

    # -------------------------
    # B) Swap MAPBoost(full) into head C3k2 blocks (P3/P4)
    # -------------------------
    targets_head = {"p3": None, "p4": None, "p5": None}

    for i, m in enumerate(layers):
        meta = get_c3k2_meta(m)
        if meta is None:
            continue
        c1, c2 = meta

        if i <= 10:
            continue

        if c1 == 256 and c2 == 64 and targets_head["p3"] is None:
            targets_head["p3"] = i
        elif c1 == 192 and c2 == 128 and targets_head["p4"] is None:
            targets_head["p4"] = i
        elif c1 == 384 and c2 == 256 and targets_head["p5"] is None:
            targets_head["p5"] = i

    print("[INFO] head C3k2 targets:", targets_head)

    swap_mapboost_idxs = []

    def swap_to_mapboost(idx: int):
        old = layers[idx]
        c1, c2 = get_c3k2_meta(old)
        new = MAPBoost(c1, c2, s=1)

        for k in ("f", "i", "type"):
            if hasattr(old, k):
                setattr(new, k, getattr(old, k))
        new.training = old.training
        layers[idx] = new
        swap_mapboost_idxs.append(idx)
        print(f"[SWAP OK] HEAD idx={idx}: C3k2({c1}->{c2}) -> MAPBoost(full)")

    if targets_head["p3"] is None or targets_head["p4"] is None:
        raise RuntimeError(f"Không tìm thấy C3k2 head P3/P4 theo meta 256->64 và 192->128. Found={targets_head}")

    swap_to_mapboost(targets_head["p3"])
    swap_to_mapboost(targets_head["p4"])

    # -------------------------
    # C) Swap WeightedConcat in place of Concat (keep Detect 3 heads)
    #   Replace Concat layers at idx: 12, 15, 18, 21 with WeightedConcat
    # -------------------------
    concat_idxs = []
    for i, m in enumerate(layers):
        if m.__class__.__name__ == "Concat":
            concat_idxs.append(i)

    # Expected: [12, 15, 18, 21] but we handle dynamically
    print("[INFO] Found Concat idxs:", concat_idxs)

    swap_wconcat_idxs = []

    def swap_to_wconcat(idx: int):
        old = layers[idx]
        # Ultralytics Concat stores dimension in old.d (usually 1)
        dim = getattr(old, "d", 1)
        # Ultralytics Concat uses old.f as sources; n_inputs = len(f)
        f = getattr(old, "f", None)
        n_inputs = len(f) if isinstance(f, (list, tuple)) else 2

        new = WeightedConcat(dimension=dim, n_inputs=n_inputs)

        for k in ("f", "i", "type"):
            if hasattr(old, k):
                setattr(new, k, getattr(old, k))
        new.training = old.training
        layers[idx] = new
        swap_wconcat_idxs.append(idx)
        print(f"[SWAP OK] Concat -> WeightedConcat at idx={idx}  (n_inputs={n_inputs}, dim={dim})")

    # You can choose which concat to replace; safest: replace all found concats
    for idx in concat_idxs:
        swap_to_wconcat(idx)

    # 4) move to device
    model.model.to(device)  # type: ignore

    # -------------------------
    # Print model info
    # -------------------------
    print("\n================ MODEL INFO (after swap) ================\n", flush=True)
    model.model.info()

    print("\n================ LAYERS (after swap) ================\n", flush=True)

    swapped_idxs = {idx_p2, idx_p3, *swap_mapboost_idxs, *swap_wconcat_idxs}

    for i, m in enumerate(model.model.model):
        name = m.__class__.__name__
        f = getattr(m, "f", None)
        np_ = sum(p.numel() for p in m.parameters()) if isinstance(m, nn.Module) else 0

        c1 = c2 = None
        if hasattr(m, "cv1") and hasattr(m, "cv2") and hasattr(m.cv1, "conv") and hasattr(m.cv2, "conv"):
            try:
                c1 = m.cv1.conv.in_channels
                c2 = m.cv2.conv.out_channels
            except Exception:
                pass

        mark = "  <== SWAPPED" if i in swapped_idxs else ""

        if c1 is not None and c2 is not None:
            print(f"{i:3d}  f={f!s:>6}  {name:<20}  c1->c2: {c1:4d}->{c2:<4d}  params={np_:>8d}{mark}")
        else:
            print(f"{i:3d}  f={f!s:>6}  {name:<20}  params={np_:>8d}{mark}")

    print("[DEBUG] model param device:", next(model.model.parameters()).device)

    # -------------------------
    # Optimizer + Data
    # -------------------------
    optimizer = Adam(model.model.parameters(), lr=cfg.lr)

    train_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "train/images"),
        labels_dir=os.path.join(cfg.base_path, "train/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )
    val_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "valid/images"),
        labels_dir=os.path.join(cfg.base_path, "valid/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )

    pin = torch.cuda.is_available()
    workers = cfg.num_workers

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )

    # -------------------------
    # Train
    # -------------------------
    train_all = []

    best_score = float("inf")
    best_epoch = -1
    best_path = os.path.join(cfg.checkpoint_dir, "best.pt")
    last_path = os.path.join(cfg.checkpoint_dir, "last.pt")

    for epoch in range(1, cfg.epochs + 1):
        print(f"\n==================== EPOCH {epoch}/{cfg.epochs} ====================")

        train_df = train_one_epoch(
            model,
            optimizer,
            train_loader,
            device,
            epoch,
            max_train_steps=cfg.max_train_steps,
            debug_print_first_batch=(epoch == 1),
        )
        train_all.append(train_df)

        if len(train_df) > 0:
            train_score = train_df[["box", "cls", "dfl"]].mean().sum()
            print(f"[EPOCH {epoch}] train_score={train_score:.6f} | best_score={best_score:.6f}")

            if train_score < best_score:
                best_score = train_score
                best_epoch = epoch
                model.save(best_path)
                print(f"[BEST] updated → epoch {best_epoch}, saved to {best_path}")
        else:
            print("[TRAIN] no valid batches produced loss (all empty labels?)")

        model.save(last_path)
        ckpt_path = os.path.join(cfg.checkpoint_dir, f"epoch_{epoch}.pt")
        model.save(ckpt_path)
        print(f"[INFO] saved checkpoint: {ckpt_path} (and last.pt)")

    train_csv = os.path.join(cfg.checkpoint_dir, "train_losses.csv")
    pd.concat(train_all, axis=0, ignore_index=True).to_csv(train_csv, index=False)
    print(f"[INFO] saved train logs: {train_csv}")

    print("\n==================== TRAIN SUMMARY ====================")
    print(f"[BEST MODEL] epoch = {best_epoch}")
    print(f"[BEST MODEL] score = {best_score:.6f}")
    print(f"[BEST MODEL] path  = {best_path}")
    print(f"[LAST MODEL] path  = {last_path}")

    # -------------------------
    # Final evaluation
    # -------------------------
    print("\n==================== FINAL EVALUATION ====================")

    eval_ckpt = best_path if os.path.exists(best_path) else last_path
    print(f"[EVAL] loading checkpoint: {eval_ckpt}")

    eval_model = YOLO(eval_ckpt)
    eval_model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore
    eval_model.model.to(device)  # type: ignore

    val_loss_df = valid_loss_only(
        eval_model,
        val_loader,
        device,
        epoch=best_epoch if os.path.exists(best_path) else cfg.epochs,
        max_val_steps=cfg.max_val_steps,
    )
    val_csv = os.path.join(cfg.checkpoint_dir, "val_losses.csv")
    val_loss_df.to_csv(val_csv, index=False)
    if len(val_loss_df) > 0:
        print("[FINAL VALID-LOSS] mean:", val_loss_df[["box", "cls", "dfl"]].mean().to_dict())
    else:
        print("[FINAL VALID-LOSS] no valid batches (empty labels?)")
    print(f"[INFO] saved val losses: {val_csv}")

    try:
        print("[FINAL VALID-METRICS] running ultralytics val() ...")
        device_id = 0 if device.type == "cuda" else "cpu"
        metrics = yolo_val_metrics(eval_model, cfg.data_yaml, device_id=device_id, workers=cfg.num_workers)

        print(f"Precision: {metrics.box.mp:.4f}")
        print(f"Recall:    {metrics.box.mr:.4f}")
        print(f"mAP@0.5:   {metrics.box.map50:.4f}")
        print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
    except Exception as e:
        print(f"[WARN] final model.val() failed: {e}")

    return model


if __name__ == "__main__":

    def running_in_notebook():
        try:
            from IPython import get_ipython

            return get_ipython() is not None
        except Exception:
            return False

    if running_in_notebook():
        print("[INFO] Running in Kaggle/Colab Notebook → using hardcoded config")
        cfg = Config(
            base_path="/kaggle/input/aod-4-yolo",
            data_yaml="/kaggle/working/data.yaml",
            epochs=4,
            batch_size=4,
            lr=3e-4,
            num_workers=2,
            max_train_steps=1000,
            max_val_steps=0,
        )
        main(cfg)
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument("--base_path", type=str, required=True)
        parser.add_argument("--data_yaml", type=str, required=True)
        parser.add_argument("--epochs", type=int, default=2)
        parser.add_argument("--batch_size", type=int, default=4)
        parser.add_argument("--lr", type=float, default=3e-4)
        parser.add_argument("--num_workers", type=int, default=2)
        parser.add_argument("--img_ext", type=str, default="jpg")
        parser.add_argument("--label_ext", type=str, default="txt")
        parser.add_argument("--checkpoint_dir", type=str, default="checkpoints")
        parser.add_argument("--max_train_steps", type=int, default=0)
        parser.add_argument("--max_val_steps", type=int, default=0)

        args = parser.parse_args()
        cfg = Config(**vars(args))
        main(cfg)

[INFO] Running in Kaggle/Colab Notebook → using hardcoded config
[INFO] device = cuda:0
[INFO] CUDA available: True
[INFO] CUDA device: Tesla P100-PCIE-16GB

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultral

train epoch 1:   0%|          | 0/3941 [00:00<?, ?it/s]

[DEBUG] images device: cuda:0
[DEBUG] targets device: cuda:0
[DEBUG] model device: cuda:0
[EPOCH 1] train_score=14.986081 | best_score=inf
[BEST] updated → epoch 1, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_1.pt (and last.pt)

==================== EPOCH 2/4 ====================


train epoch 2:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 2] train_score=11.103099 | best_score=14.986081
[BEST] updated → epoch 2, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_2.pt (and last.pt)

==================== EPOCH 3/4 ====================


...

[EPOCH 3] train_score=10.037219 | best_score=11.103099
[BEST] updated → epoch 3, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_3.pt (and last.pt)

==================== EPOCH 4/4 ====================


train epoch 4:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 4] train_score=9.284618 | best_score=10.037219
[BEST] updated → epoch 4, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_4.pt (and last.pt)
[INFO] saved train logs: checkpoints/train_losses.csv

==================== TRAIN SUMMARY ====================
[BEST MODEL] epoch = 4
[BEST MODEL] score = 9.284618
[BEST MODEL] path  = checkpoints/best.pt
[LAST MODEL] path  = checkpoints/last.pt

==================== FINAL EVALUATION ====================
[EVAL] loading checkpoint: checkpoints/best.pt


valid(loss) epoch 4:   0%|          | 0/1129 [00:00<?, ?it/s]

[FINAL VALID-LOSS] mean: {'box': 3.9951542809405134, 'cls': 2.187780405801508, 'dfl': 3.4079484670151508}
[INFO] saved val losses: checkpoints/val_losses.csv
[FINAL VALID-METRICS] running ultralytics val() ...
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
custom_YOLOv11n summary: 127 layers, 2,888,276 parameters, 879,644 gradients, 8.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 45.0±26.6 MB/s, size: 47.2 KB)
val: Scanning /kaggle/input/aod-4-yolo/valid/labels... 4514 images, 125 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4514/4514 895.7it/s 5.0s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/aod-4-yolo/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 283/283 10.5it/s 26.9s0.2ss
                   all       4514       6369       0.28       0.29      0.214     0.0799
Speed: 0.8ms preprocess, 2.7ms inference, 0.0ms loss, 0.8ms po

# Bước 3 – Sửa Detect thành single head
# Trước	Sau
# Detect(f=[16,19,22])	Detect(f=[NEW])

# Trong đó:

# NEW = index của MAPBoost full (192→128)

In [23]:
%%writefile hoangtn_single_detect.py

import torch
import torch.nn as nn

from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.modules.head import Detect

from hoangtn_WeightedConcat import WeightedConcat
from hoangtn_MAPBoost import MAPBoost


class PaperSingleDetectHead(nn.Module):
    """
    Paper-style single detect head
    Fully compatible with Ultralytics v8DetectionLoss
    """

    def __init__(self, nc: int, reg_max: int = 16, stride: int = 8):
        super().__init__()

        # ===============================
        # Required metadata for Ultralytics
        # ===============================
        self.nc = nc
        self.reg_max = reg_max
        self.no = nc + reg_max * 4
        self.nl = 1
        self.stride = torch.tensor([stride], dtype=torch.float)
        self.anchors = None

        # ===============================
        # Paper architecture
        # ===============================

        # P3 (64ch) -> P4 resolution
        self.p3_down = Conv(64, 64, k=3, s=2)

        # Weighted fusion (P3↓ + P4)
        self.fuse = WeightedConcat(dimension=1, n_inputs=2)  # 64 + 128 = 192

        # MAPBoost refinement
        self.mapboost = MAPBoost(192, 128, s=1)

        # Ultralytics Detect (single scale)
        self.detect = Detect(nc, reg_max, None, [128])

    def forward(self, x):
        """
        x: list of feature maps [P3, P4, P5]
        """
        p3, p4 = x[0], x[1]

        p3d = self.p3_down(p3)
        fused = self.fuse([p3d, p4])
        fused = self.mapboost(fused)

        return self.detect([fused])


Writing hoangtn_single_detect.py


In [26]:
# train_loop.py
import argparse
from dataclasses import dataclass

import pandas as pd
import torch

############################################################
# Custom modules
from hoangtn_single_detect import PaperSingleDetectHead
from torch import Tensor
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset

from ultralytics import YOLO
from ultralytics.nn.modules.conv import Conv

############################################################


# -----------------------------
# Utils
# -----------------------------
def get_device() -> torch.device:
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


# -----------------------------
# Dataset
# -----------------------------
class AodDataset(Dataset):
    def __init__(
        self,
        images_dir: str,
        labels_dir: str,
        images_ext: str = "jpg",
        labels_ext: str = "txt",
        transforms=None,
        verbose_bad_labels: bool = False,
    ):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.images_ext = images_ext
        self.labels_ext = labels_ext
        self.transforms = transforms
        self.verbose_bad_labels = verbose_bad_labels

        img_glob = os.path.join(images_dir, f"*.{images_ext}")
        self.images = sorted(glob.glob(img_glob))
        if len(self.images) == 0:
            raise FileNotFoundError(f"No images found: {img_glob}")

        self.miss_labeled: list[str] = []

    def _labels_path_from_image(self, image_path: str) -> str:
        base = os.path.basename(image_path)
        stem = base[: -(len(self.images_ext) + 1)]
        return os.path.join(self.labels_dir, f"{stem}.{self.labels_ext}")

    def _read_labels(self, labels_path: str) -> Tensor:
        if not os.path.exists(labels_path):
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        labels: list[list[float]] = []
        with open(labels_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) != 5:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid values count: {line}")
                    continue
                try:
                    c, xc, yc, w, h = map(float, parts)
                except ValueError:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: cannot parse: {line}")
                    continue

                if not (0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid bbox: {line}")
                    continue

                labels.append([c, xc, yc, w, h])

        if len(labels) == 0:
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        return torch.tensor(labels, dtype=torch.float32)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        image_path = self.images[idx]
        img = decode_image(image_path)  # uint8 [C,H,W] on CPU
        if self.transforms is not None:
            img = self.transforms(img)

        labels_path = self._labels_path_from_image(image_path)
        targets = self._read_labels(labels_path)
        return img, targets

    def __len__(self) -> int:
        return len(self.images)


# -----------------------------
# Collate & Batch builder
# -----------------------------
def collate_fn(batch: list[tuple[Tensor, Tensor]]) -> tuple[Tensor, Tensor]:
    images, targets = zip(*batch)
    images = torch.stack(images, dim=0)
    padded_targets = pad_sequence(list(targets), batch_first=True, padding_value=-1.0)
    return images, padded_targets


def prepare_batch_dict(images: Tensor, targets: Tensor) -> dict:
    valid_mask = targets[..., 0] != -1
    if not valid_mask.any():
        return {
            "img": images,
            "batch_idx": torch.empty((0,), device=images.device, dtype=torch.int64),
            "cls": torch.empty((0,), device=images.device, dtype=torch.float32),
            "bboxes": torch.empty((0, 4), device=images.device, dtype=torch.float32),
        }

    valid_targets = targets[valid_mask]  # [M,5]

    per_img_counts = valid_mask.sum(dim=1).to(torch.int64)
    batch_idx_list = []
    for i, cnt in enumerate(per_img_counts.tolist()):
        if cnt > 0:
            batch_idx_list.append(torch.full((cnt,), i, device=images.device, dtype=torch.int64))
    batch_idx = (
        torch.cat(batch_idx_list, dim=0)
        if batch_idx_list
        else torch.empty((0,), device=images.device, dtype=torch.int64)
    )

    cls = valid_targets[:, 0].to(torch.float32)
    bboxes = valid_targets[:, 1:].to(torch.float32)
    return {"img": images, "batch_idx": batch_idx, "cls": cls, "bboxes": bboxes}


# -----------------------------
# Train / Valid
# -----------------------------
def train_one_epoch(
    model: YOLO,
    optimizer: Adam,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_train_steps: int = 0,
    debug_print_first_batch: bool = True,
) -> pd.DataFrame:
    model.model.train(True)
    torch.set_grad_enabled(True)
    loss_fn = v8DetectionLoss(model.model)

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"train epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_train_steps > 0 and step >= max_train_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        if debug_print_first_batch and step == 0:
            print("[DEBUG] images device:", images.device)
            print("[DEBUG] targets device:", targets.device)
            print("[DEBUG] model device:", next(model.model.parameters()).device)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            pbar.set_postfix_str("skip(empty)")
            continue

        optimizer.zero_grad(set_to_none=True)

        with torch.inference_mode(False):
            with torch.enable_grad():
                preds = model.model(images)
                batch_loss, last_loss = loss_fn(preds, batch)
                loss_scalar = batch_loss.sum()

        if not torch.isfinite(loss_scalar):
            pbar.set_postfix_str("skip(non-finite)")
            continue

        if not loss_scalar.requires_grad:
            raise RuntimeError(
                f"loss_scalar.requires_grad=False at step {step}. "
                f"Grad is disabled globally or inference_mode is active."
            )

        loss_scalar.backward()
        optimizer.step()

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        row = {"epoch": epoch, "step": step, "box": float(box_loss), "cls": float(cls_loss), "dfl": float(dfl_loss)}
        rows.append(row)
        pbar.set_postfix({"box": f"{row['box']:.4f}", "cls": f"{row['cls']:.4f}", "dfl": f"{row['dfl']:.4f}"})

    return pd.DataFrame(rows)


@torch.no_grad()
def valid_loss_only(
    model: YOLO,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_val_steps: int = 0,
) -> pd.DataFrame:
    model.model.eval()
    loss_fn = v8DetectionLoss(model.model)

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"valid(loss) epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_val_steps > 0 and step >= max_val_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            continue

        preds = model.model(images)
        _, last_loss = loss_fn(preds, batch)

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        rows.append(
            {"epoch": epoch, "batch": step, "box": float(box_loss), "cls": float(cls_loss), "dfl": float(dfl_loss)}
        )

    return pd.DataFrame(rows)


def yolo_val_metrics(model: YOLO, data_yaml: str, device_id=0, workers: int = 0):
    return model.val(data=data_yaml, device=device_id, workers=workers, verbose=False, plots=False)


# -----------------------------
# Main
# -----------------------------
@dataclass
class Config:
    base_path: str
    data_yaml: str
    epochs: int = 2
    batch_size: int = 4
    lr: float = 3e-4
    num_workers: int = 2
    img_ext: str = "jpg"
    label_ext: str = "txt"
    checkpoint_dir: str = "checkpoints"
    max_train_steps: int = 0
    max_val_steps: int = 0


def main(cfg: Config):
    device = get_device()
    print(f"[INFO] device = {device}")
    if device.type == "cuda":
        print("[INFO] CUDA available:", torch.cuda.is_available())
        print("[INFO] CUDA device:", torch.cuda.get_device_name(0))

    ensure_dir(cfg.checkpoint_dir)

    # 1) load YOLO wrapper
    model = YOLO(model="custom_yolov11n.yaml", task="detect", verbose=True)

    # 2) set loss weights
    model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore

    # 3) SWAP layers
    layers = model.model.model  # nn.ModuleList

    def is_ultra_conv(m: nn.Module) -> bool:
        return hasattr(m, "conv") and isinstance(getattr(m, "conv", None), nn.Conv2d)

    def get_conv_meta(m: nn.Module):
        if is_ultra_conv(m):
            conv = m.conv
            s = conv.stride[0] if isinstance(conv.stride, tuple) else int(conv.stride)
            return conv.in_channels, conv.out_channels, s
        return None

    def get_c3k2_meta(m: nn.Module):
        if hasattr(m, "cv1") and hasattr(m, "cv2") and hasattr(m.cv1, "conv") and hasattr(m.cv2, "conv"):
            try:
                return m.cv1.conv.in_channels, m.cv2.conv.out_channels
            except Exception:
                return None
        return None

    # -------------------------
    # A) Swap MAPBoostLite into backbone downsample convs
    # -------------------------
    targets = {"p2": None, "p3": None}

    for i, m in enumerate(layers):
        meta = get_conv_meta(m)
        if meta is None:
            continue
        c1, c2, s = meta
        if s == 2 and c2 == 128 and targets["p2"] is None:
            targets["p2"] = i
        if s == 2 and c2 == 256 and targets["p3"] is None:
            targets["p3"] = i

    if targets["p2"] is None or targets["p3"] is None:
        print("\n[DEBUG] List Conv(stride=2):")
        for i, m in enumerate(layers):
            meta = get_conv_meta(m)
            if meta is None:
                continue
            c1, c2, s = meta
            if s == 2:
                print(f"  idx={i:3d}  Conv  {c1}->{c2}  stride={s}")
        raise RuntimeError(f"Không tìm thấy Conv stride=2 cho P2/4(out=128) hoặc P3/8(out=256). Found={targets}")

    idx_p2 = targets["p2"]
    old_p2 = layers[idx_p2]
    c1_p2, c2_p2, _ = get_conv_meta(old_p2)
    new_p2 = MAPBoostLite(c1_p2, c2_p2, k=3, s=2, normalize=True)
    for k in ("f", "i", "type"):
        if hasattr(old_p2, k):
            setattr(new_p2, k, getattr(old_p2, k))
    new_p2.training = old_p2.training
    layers[idx_p2] = new_p2
    print(f"[SWAP OK] P2/4 at idx={idx_p2}: Conv({c1_p2}->{c2_p2}, s=2) -> MAPBoostLite")

    idx_p3 = targets["p3"]
    old_p3 = layers[idx_p3]
    c1_p3, c2_p3, _ = get_conv_meta(old_p3)
    new_p3 = MAPBoostLite(c1_p3, c2_p3, k=3, s=2, normalize=True)
    for k in ("f", "i", "type"):
        if hasattr(old_p3, k):
            setattr(new_p3, k, getattr(old_p3, k))
    new_p3.training = old_p3.training
    layers[idx_p3] = new_p3
    print(f"[SWAP OK] P3/8 at idx={idx_p3}: Conv({c1_p3}->{c2_p3}, s=2) -> MAPBoostLite")

    # -------------------------
    # B) Swap MAPBoost(full) into head C3k2 blocks (P3/P4)
    # -------------------------
    targets_head = {"p3": None, "p4": None, "p5": None}

    for i, m in enumerate(layers):
        meta = get_c3k2_meta(m)
        if meta is None:
            continue
        c1, c2 = meta
        if i <= 10:
            continue
        if c1 == 256 and c2 == 64 and targets_head["p3"] is None:
            targets_head["p3"] = i
        elif c1 == 192 and c2 == 128 and targets_head["p4"] is None:
            targets_head["p4"] = i
        elif c1 == 384 and c2 == 256 and targets_head["p5"] is None:
            targets_head["p5"] = i

    print("[INFO] head C3k2 targets:", targets_head)

    swap_mapboost_idxs = []

    def swap_to_mapboost(idx: int):
        old = layers[idx]
        c1, c2 = get_c3k2_meta(old)
        new = MAPBoost(c1, c2, s=1)
        for k in ("f", "i", "type"):
            if hasattr(old, k):
                setattr(new, k, getattr(old, k))
        new.training = old.training
        layers[idx] = new
        swap_mapboost_idxs.append(idx)
        print(f"[SWAP OK] HEAD idx={idx}: C3k2({c1}->{c2}) -> MAPBoost(full)")

    if targets_head["p3"] is None or targets_head["p4"] is None:
        raise RuntimeError(f"Không tìm thấy C3k2 head P3/P4 theo meta 256->64 và 192->128. Found={targets_head}")

    swap_to_mapboost(targets_head["p3"])
    swap_to_mapboost(targets_head["p4"])

    # -------------------------
    # C) WeightedConcat in place of Concat (optional but consistent)
    # -------------------------
    concat_idxs = []
    for i, m in enumerate(layers):
        if m.__class__.__name__ == "Concat":
            concat_idxs.append(i)
    print("[INFO] Found Concat idxs:", concat_idxs)

    swap_wconcat_idxs = []

    def swap_to_wconcat(idx: int):
        old = layers[idx]
        dim = getattr(old, "d", 1)
        f = getattr(old, "f", None)
        n_inputs = len(f) if isinstance(f, (list, tuple)) else 2
        new = WeightedConcat(dimension=dim, n_inputs=n_inputs)
        for k in ("f", "i", "type"):
            if hasattr(old, k):
                setattr(new, k, getattr(old, k))
        new.training = old.training
        layers[idx] = new
        swap_wconcat_idxs.append(idx)
        print(f"[SWAP OK] Concat -> WeightedConcat at idx={idx}  (n_inputs={n_inputs}, dim={dim})")

    for idx in concat_idxs:
        swap_to_wconcat(idx)

    # --------------------------------------------------
    # D) Swap Detect -> PaperSingleDetectHead (1-head)
    # --------------------------------------------------

    detect_idx = None
    for i, m in enumerate(layers):
        if m.__class__.__name__ == "Detect":
            detect_idx = i
            break

    if detect_idx is None:
        raise RuntimeError("Detect layer not found")

    old_detect = layers[detect_idx]

    # number of classes from original Detect
    nc = old_detect.nc
    reg_max = old_detect.reg_max

    new_detect = PaperSingleDetectHead(
        nc=nc,
        reg_max=reg_max,
        stride=8,  # P4 resolution
    )

    # copy Ultralytics graph metadata
    for k in ("f", "i", "type"):
        if hasattr(old_detect, k):
            setattr(new_detect, k, getattr(old_detect, k))

    new_detect.training = old_detect.training
    layers[detect_idx] = new_detect

    print(f"[SWAP OK] Detect(3-head) -> PaperSingleDetectHead at idx={detect_idx}")

    # 4) move to device
    model.model.to(device)  # type: ignore

    # -------------------------
    # Print model info
    # -------------------------
    print("\n================ MODEL INFO (after swap) ================\n", flush=True)
    model.model.info()

    print("\n================ LAYERS (after swap) ================\n", flush=True)

    swapped_idxs = {idx_p2, idx_p3, *swap_mapboost_idxs, *swap_wconcat_idxs, detect_idx}

    for i, m in enumerate(model.model.model):
        name = m.__class__.__name__
        f = getattr(m, "f", None)
        np_ = sum(p.numel() for p in m.parameters()) if isinstance(m, nn.Module) else 0

        c1 = c2 = None
        if hasattr(m, "cv1") and hasattr(m, "cv2") and hasattr(m.cv1, "conv") and hasattr(m.cv2, "conv"):
            try:
                c1 = m.cv1.conv.in_channels
                c2 = m.cv2.conv.out_channels
            except Exception:
                pass

        mark = "  <== SWAPPED" if i in swapped_idxs else ""

        if c1 is not None and c2 is not None:
            print(f"{i:3d}  f={f!s:>6}  {name:<24}  c1->c2: {c1:4d}->{c2:<4d}  params={np_:>8d}{mark}")
        else:
            print(f"{i:3d}  f={f!s:>6}  {name:<24}  params={np_:>8d}{mark}")

    print("[DEBUG] model param device:", next(model.model.parameters()).device)

    # -------------------------
    # Optimizer + Data
    # -------------------------
    optimizer = Adam(model.model.parameters(), lr=cfg.lr)

    train_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "train/images"),
        labels_dir=os.path.join(cfg.base_path, "train/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )
    val_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "valid/images"),
        labels_dir=os.path.join(cfg.base_path, "valid/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )

    pin = torch.cuda.is_available()
    workers = cfg.num_workers

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )

    # -------------------------
    # Train
    # -------------------------
    train_all = []

    best_score = float("inf")
    best_epoch = -1
    best_path = os.path.join(cfg.checkpoint_dir, "best.pt")
    last_path = os.path.join(cfg.checkpoint_dir, "last.pt")

    for epoch in range(1, cfg.epochs + 1):
        print(f"\n==================== EPOCH {epoch}/{cfg.epochs} ====================")

        train_df = train_one_epoch(
            model,
            optimizer,
            train_loader,
            device,
            epoch,
            max_train_steps=cfg.max_train_steps,
            debug_print_first_batch=(epoch == 1),
        )
        train_all.append(train_df)

        if len(train_df) > 0:
            train_score = train_df[["box", "cls", "dfl"]].mean().sum()
            print(f"[EPOCH {epoch}] train_score={train_score:.6f} | best_score={best_score:.6f}")

            if train_score < best_score:
                best_score = train_score
                best_epoch = epoch
                model.save(best_path)
                print(f"[BEST] updated → epoch {best_epoch}, saved to {best_path}")
        else:
            print("[TRAIN] no valid batches produced loss (all empty labels?)")

        model.save(last_path)
        ckpt_path = os.path.join(cfg.checkpoint_dir, f"epoch_{epoch}.pt")
        model.save(ckpt_path)
        print(f"[INFO] saved checkpoint: {ckpt_path} (and last.pt)")

    train_csv = os.path.join(cfg.checkpoint_dir, "train_losses.csv")
    pd.concat(train_all, axis=0, ignore_index=True).to_csv(train_csv, index=False)
    print(f"[INFO] saved train logs: {train_csv}")

    print("\n==================== TRAIN SUMMARY ====================")
    print(f"[BEST MODEL] epoch = {best_epoch}")
    print(f"[BEST MODEL] score = {best_score:.6f}")
    print(f"[BEST MODEL] path  = {best_path}")
    print(f"[LAST MODEL] path  = {last_path}")

    # -------------------------
    # Final evaluation
    # -------------------------
    print("\n==================== FINAL EVALUATION ====================")

    eval_ckpt = best_path if os.path.exists(best_path) else last_path
    print(f"[EVAL] loading checkpoint: {eval_ckpt}")

    eval_model = YOLO(eval_ckpt)
    eval_model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore
    eval_model.model.to(device)  # type: ignore

    val_loss_df = valid_loss_only(
        eval_model,
        val_loader,
        device,
        epoch=best_epoch if os.path.exists(best_path) else cfg.epochs,
        max_val_steps=cfg.max_val_steps,
    )
    val_csv = os.path.join(cfg.checkpoint_dir, "val_losses.csv")
    val_loss_df.to_csv(val_csv, index=False)
    if len(val_loss_df) > 0:
        print("[FINAL VALID-LOSS] mean:", val_loss_df[["box", "cls", "dfl"]].mean().to_dict())
    else:
        print("[FINAL VALID-LOSS] no valid batches (empty labels?)")
    print(f"[INFO] saved val losses: {val_csv}")

    try:
        print("[FINAL VALID-METRICS] running ultralytics val() ...")
        device_id = 0 if device.type == "cuda" else "cpu"
        metrics = yolo_val_metrics(eval_model, cfg.data_yaml, device_id=device_id, workers=cfg.num_workers)

        print(f"Precision: {metrics.box.mp:.4f}")
        print(f"Recall:    {metrics.box.mr:.4f}")
        print(f"mAP@0.5:   {metrics.box.map50:.4f}")
        print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
    except Exception as e:
        print(f"[WARN] final model.val() failed: {e}")

    return model


if __name__ == "__main__":

    def running_in_notebook():
        try:
            from IPython import get_ipython

            return get_ipython() is not None
        except Exception:
            return False

    if running_in_notebook():
        print("[INFO] Running in Kaggle/Colab Notebook → using hardcoded config")
        cfg = Config(
            base_path="/kaggle/input/aod-4-yolo",
            data_yaml="/kaggle/working/data.yaml",
            epochs=4,
            batch_size=4,
            lr=3e-4,
            num_workers=2,
            max_train_steps=1000,
            max_val_steps=0,
        )
        main(cfg)
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument("--base_path", type=str, required=True)
        parser.add_argument("--data_yaml", type=str, required=True)
        parser.add_argument("--epochs", type=int, default=2)
        parser.add_argument("--batch_size", type=int, default=4)
        parser.add_argument("--lr", type=float, default=3e-4)
        parser.add_argument("--num_workers", type=int, default=2)
        parser.add_argument("--img_ext", type=str, default="jpg")
        parser.add_argument("--label_ext", type=str, default="txt")
        parser.add_argument("--checkpoint_dir", type=str, default="checkpoints")
        parser.add_argument("--max_train_steps", type=int, default=0)
        parser.add_argument("--max_val_steps", type=int, default=0)

        args = parser.parse_args()
        cfg = Config(**vars(args))
        main(cfg)

[INFO] Running in Kaggle/Colab Notebook → using hardcoded config
[INFO] device = cuda:0
[INFO] CUDA available: True
[INFO] CUDA device: Tesla P100-PCIE-16GB

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultral

train epoch 1:   0%|          | 0/3941 [00:00<?, ?it/s]

[DEBUG] images device: cuda:0
[DEBUG] targets device: cuda:0
[DEBUG] model device: cuda:0
[EPOCH 1] train_score=602.325318 | best_score=inf
[BEST] updated → epoch 1, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_1.pt (and last.pt)

==================== EPOCH 2/4 ====================


train epoch 2:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 2] train_score=32.048220 | best_score=602.325318
[BEST] updated → epoch 2, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_2.pt (and last.pt)

==================== EPOCH 3/4 ====================


...

[EPOCH 3] train_score=16.588388 | best_score=32.048220
[BEST] updated → epoch 3, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_3.pt (and last.pt)

==================== EPOCH 4/4 ====================


train epoch 4:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 4] train_score=12.449489 | best_score=16.588388
[BEST] updated → epoch 4, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_4.pt (and last.pt)
[INFO] saved train logs: checkpoints/train_losses.csv

==================== TRAIN SUMMARY ====================
[BEST MODEL] epoch = 4
[BEST MODEL] score = 12.449489
[BEST MODEL] path  = checkpoints/best.pt
[LAST MODEL] path  = checkpoints/last.pt

==================== FINAL EVALUATION ====================
[EVAL] loading checkpoint: checkpoints/best.pt


valid(loss) epoch 4:   0%|          | 0/1129 [00:00<?, ?it/s]

[FINAL VALID-LOSS] mean: {'box': 4.678316030989407, 'cls': 5.495866234206298, 'dfl': 2.9539018982755123}
[INFO] saved val losses: checkpoints/val_losses.csv
[FINAL VALID-METRICS] running ultralytics val() ...
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
custom_YOLOv11n summary: 128 layers, 2,904,594 parameters, 1,129,498 gradients, 8.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 35.0±11.1 MB/s, size: 40.4 KB)
val: Scanning /kaggle/input/aod-4-yolo/valid/labels... 4514 images, 125 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4514/4514 585.0it/s 7.7s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/aod-4-yolo/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 283/283 10.0it/s 28.4s0.1ss
                   all       4514       6369          0          0          0          0
Speed: 0.7ms preprocess, 2.6ms inference, 0.0ms loss, 1.5ms p

# Final Test # 
# Code lỗi do Valid ultralytics không hỗ trợ tắt detect thành 1  => Cách làm thay thế là chạy giữ nguyên 3 head tuy nhiên nhân trọng số làm những head detect không mong muốn về gần 0

# Layer 22  thay C3k2 bằng 22 f=    -1  Conv                  params=   98816  <== SWAPPED => giảm layer cũng như là kích thước model không ảnh hưởng tới 2 detect trên cho vật thể nhỏ mà giảm được kích thước và tốc độ

Tại sao thay bằng Conv lại “đúng ý” pseudo-single-head?

C3k2 là block nặng (nhiều conv + bottleneck) → nhiều than số + nhiều “layers” nội bộ

Conv 1x1 là block nhẹ → vẫn giữ đúng shape 384→256 để Detect không gãy, nhưng làm P5 “auxiliary” đúng mục tiêu.

In [27]:
# train_loop.py
import argparse
from dataclasses import dataclass

import pandas as pd
import torch
from torch import Tensor
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset

from ultralytics import YOLO

############################################################
# Custom modules

############################################################


# -----------------------------
# Utils
# -----------------------------
def get_device() -> torch.device:
    # Force cuda:0 on Kaggle when available
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


# -----------------------------
# Dataset
# -----------------------------
class AodDataset(Dataset):
    """
    Read images + YOLO labels (cls xc yc w h) normalized [0..1].

    Return:
      - image: uint8 tensor [C,H,W] on CPU
      - targets: float tensor [N,5] on CPU (or [[-1]*5] if empty/missing).
    """

    def __init__(
        self,
        images_dir: str,
        labels_dir: str,
        images_ext: str = "jpg",
        labels_ext: str = "txt",
        transforms=None,
        verbose_bad_labels: bool = False,
    ):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.images_ext = images_ext
        self.labels_ext = labels_ext
        self.transforms = transforms
        self.verbose_bad_labels = verbose_bad_labels

        img_glob = os.path.join(images_dir, f"*.{images_ext}")
        self.images = sorted(glob.glob(img_glob))
        if len(self.images) == 0:
            raise FileNotFoundError(f"No images found: {img_glob}")

        self.miss_labeled: list[str] = []

    def _labels_path_from_image(self, image_path: str) -> str:
        base = os.path.basename(image_path)
        stem = base[: -(len(self.images_ext) + 1)]
        return os.path.join(self.labels_dir, f"{stem}.{self.labels_ext}")

    def _read_labels(self, labels_path: str) -> Tensor:
        if not os.path.exists(labels_path):
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        labels: list[list[float]] = []
        with open(labels_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) != 5:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid values count: {line}")
                    continue
                try:
                    c, xc, yc, w, h = map(float, parts)
                except ValueError:
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: cannot parse: {line}")
                    continue

                if not (0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
                    if self.verbose_bad_labels:
                        print(f"[WARN] {labels_path}: invalid bbox: {line}")
                    continue

                labels.append([c, xc, yc, w, h])

        if len(labels) == 0:
            self.miss_labeled.append(labels_path)
            return torch.tensor([[-1, -1, -1, -1, -1]], dtype=torch.float32)

        return torch.tensor(labels, dtype=torch.float32)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        image_path = self.images[idx]
        img = decode_image(image_path)  # uint8 [C,H,W] on CPU
        if self.transforms is not None:
            img = self.transforms(img)

        labels_path = self._labels_path_from_image(image_path)
        targets = self._read_labels(labels_path)
        return img, targets

    def __len__(self) -> int:
        return len(self.images)


# -----------------------------
# Collate & Batch builder
# -----------------------------
def collate_fn(batch: list[tuple[Tensor, Tensor]]) -> tuple[Tensor, Tensor]:
    images, targets = zip(*batch)
    images = torch.stack(images, dim=0)
    padded_targets = pad_sequence(list(targets), batch_first=True, padding_value=-1.0)
    return images, padded_targets


def prepare_batch_dict(images: Tensor, targets: Tensor) -> dict:
    valid_mask = targets[..., 0] != -1
    if not valid_mask.any():
        return {
            "img": images,
            "batch_idx": torch.empty((0,), device=images.device, dtype=torch.int64),
            "cls": torch.empty((0,), device=images.device, dtype=torch.float32),
            "bboxes": torch.empty((0, 4), device=images.device, dtype=torch.float32),
        }

    valid_targets = targets[valid_mask]  # [M,5]

    per_img_counts = valid_mask.sum(dim=1).to(torch.int64)
    batch_idx_list = []
    for i, cnt in enumerate(per_img_counts.tolist()):
        if cnt > 0:
            batch_idx_list.append(torch.full((cnt,), i, device=images.device, dtype=torch.int64))
    batch_idx = (
        torch.cat(batch_idx_list, dim=0)
        if batch_idx_list
        else torch.empty((0,), device=images.device, dtype=torch.int64)
    )

    cls = valid_targets[:, 0].to(torch.float32)
    bboxes = valid_targets[:, 1:].to(torch.float32)
    return {"img": images, "batch_idx": batch_idx, "cls": cls, "bboxes": bboxes}


# -----------------------------
# Train / Valid
# -----------------------------
def train_one_epoch(
    model: YOLO,
    optimizer: Adam,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_train_steps: int = 0,
    debug_print_first_batch: bool = True,
) -> pd.DataFrame:
    model.model.train(True)
    torch.set_grad_enabled(True)
    loss_fn = v8DetectionLoss(model.model)

    # PSEUDO-SINGLE-HEAD (paper-friendly) loss weights:
    # P3 strongest, P4 medium, P5 weakest
    # => total_loss = 1.0*loss(P3) + 0.8*loss(P4) + 0.2*loss(P5)
    if hasattr(loss_fn, "balance"):
        loss_fn.balance = [1.0, 0.8, 0.2]

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"train epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_train_steps > 0 and step >= max_train_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        if debug_print_first_batch and step == 0:
            print("[DEBUG] images device:", images.device)
            print("[DEBUG] targets device:", targets.device)
            print("[DEBUG] model device:", next(model.model.parameters()).device)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            pbar.set_postfix_str("skip(empty)")
            continue

        optimizer.zero_grad(set_to_none=True)

        with torch.inference_mode(False):
            with torch.enable_grad():
                preds = model.model(images)
                batch_loss, last_loss = loss_fn(preds, batch)
                loss_scalar = batch_loss.sum()

        if not torch.isfinite(loss_scalar):
            pbar.set_postfix_str("skip(non-finite)")
            continue

        if not loss_scalar.requires_grad:
            raise RuntimeError(
                f"loss_scalar.requires_grad=False at step {step}. "
                f"Grad is disabled globally or inference_mode is active."
            )

        loss_scalar.backward()
        optimizer.step()

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        row = {"epoch": epoch, "step": step, "box": float(box_loss), "cls": float(cls_loss), "dfl": float(dfl_loss)}
        rows.append(row)
        pbar.set_postfix({"box": f"{row['box']:.4f}", "cls": f"{row['cls']:.4f}", "dfl": f"{row['dfl']:.4f}"})

    return pd.DataFrame(rows)


@torch.no_grad()
def valid_loss_only(
    model: YOLO,
    dataloader: DataLoader,
    device: torch.device,
    epoch: int,
    max_val_steps: int = 0,
) -> pd.DataFrame:
    model.model.eval()
    loss_fn = v8DetectionLoss(model.model)

    # same balance during val-loss (optional but consistent)
    if hasattr(loss_fn, "balance"):
        loss_fn.balance = [1.0, 0.8, 0.2]

    rows = []
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"valid(loss) epoch {epoch}")

    for step, (images, targets) in pbar:
        if max_val_steps > 0 and step >= max_val_steps:
            break

        images = images.to(device, non_blocking=True).float() / 255.0
        targets = targets.to(device, non_blocking=True)

        batch = prepare_batch_dict(images, targets)
        if batch["bboxes"].numel() == 0:
            continue

        preds = model.model(images)
        _, last_loss = loss_fn(preds, batch)

        box_loss, cls_loss, dfl_loss = last_loss.detach().cpu().tolist()
        rows.append(
            {"epoch": epoch, "batch": step, "box": float(box_loss), "cls": float(cls_loss), "dfl": float(dfl_loss)}
        )

    return pd.DataFrame(rows)


def yolo_val_metrics(model: YOLO, data_yaml: str, device_id=0, workers: int = 0):
    return model.val(data=data_yaml, device=device_id, workers=workers, verbose=False, plots=False)


# -----------------------------
# Main
# -----------------------------
@dataclass
class Config:
    base_path: str
    data_yaml: str
    epochs: int = 2
    batch_size: int = 4
    lr: float = 3e-4
    num_workers: int = 2
    img_ext: str = "jpg"
    label_ext: str = "txt"
    checkpoint_dir: str = "checkpoints"
    max_train_steps: int = 0
    max_val_steps: int = 0


def main(cfg: Config):
    device = get_device()
    print(f"[INFO] device = {device}")
    if device.type == "cuda":
        print("[INFO] CUDA available:", torch.cuda.is_available())
        print("[INFO] CUDA device:", torch.cuda.get_device_name(0))

    ensure_dir(cfg.checkpoint_dir)

    # 1) load YOLO wrapper
    model = YOLO(model="custom_yolov11n.yaml", task="detect", verbose=True)

    # 2) set loss weights (Ultralytics hyp)
    model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore

    # 3) SWAP layers
    layers = model.model.model  # nn.ModuleList

    def is_ultra_conv(m: nn.Module) -> bool:
        return hasattr(m, "conv") and isinstance(getattr(m, "conv", None), nn.Conv2d)

    def get_conv_meta(m: nn.Module):
        if is_ultra_conv(m):
            conv = m.conv
            s = conv.stride[0] if isinstance(conv.stride, tuple) else int(conv.stride)
            return conv.in_channels, conv.out_channels, s
        return None

    def get_c3k2_meta(m: nn.Module):
        if hasattr(m, "cv1") and hasattr(m, "cv2") and hasattr(m.cv1, "conv") and hasattr(m.cv2, "conv"):
            try:
                return m.cv1.conv.in_channels, m.cv2.conv.out_channels
            except Exception:
                return None
        return None

    # -------------------------
    # A) Swap MAPBoostLite into backbone downsample convs
    # -------------------------
    targets = {"p2": None, "p3": None}

    for i, m in enumerate(layers):
        meta = get_conv_meta(m)
        if meta is None:
            continue
        c1, c2, s = meta
        if s == 2 and c2 == 128 and targets["p2"] is None:
            targets["p2"] = i
        if s == 2 and c2 == 256 and targets["p3"] is None:
            targets["p3"] = i

    if targets["p2"] is None or targets["p3"] is None:
        print("\n[DEBUG] List Conv(stride=2):")
        for i, m in enumerate(layers):
            meta = get_conv_meta(m)
            if meta is None:
                continue
            c1, c2, s = meta
            if s == 2:
                print(f"  idx={i:3d}  Conv  {c1}->{c2}  stride={s}")
        raise RuntimeError(f"Không tìm thấy Conv stride=2 cho P2/4(out=128) hoặc P3/8(out=256). Found={targets}")

    # Swap P2/4
    idx_p2 = targets["p2"]
    old_p2 = layers[idx_p2]
    c1_p2, c2_p2, _ = get_conv_meta(old_p2)
    new_p2 = MAPBoostLite(c1_p2, c2_p2, k=3, s=2, normalize=True)
    for k in ("f", "i", "type"):
        if hasattr(old_p2, k):
            setattr(new_p2, k, getattr(old_p2, k))
    new_p2.training = old_p2.training
    layers[idx_p2] = new_p2
    print(f"[SWAP OK] P2/4 at idx={idx_p2}: Conv({c1_p2}->{c2_p2}, s=2) -> MAPBoostLite")

    # Swap P3/8
    idx_p3 = targets["p3"]
    old_p3 = layers[idx_p3]
    c1_p3, c2_p3, _ = get_conv_meta(old_p3)
    new_p3 = MAPBoostLite(c1_p3, c2_p3, k=3, s=2, normalize=True)
    for k in ("f", "i", "type"):
        if hasattr(old_p3, k):
            setattr(new_p3, k, getattr(old_p3, k))
    new_p3.training = old_p3.training
    layers[idx_p3] = new_p3
    print(f"[SWAP OK] P3/8 at idx={idx_p3}: Conv({c1_p3}->{c2_p3}, s=2) -> MAPBoostLite")

    # -------------------------
    # B) Swap MAPBoost(full) into head C3k2 blocks (P3/P4)
    #    AND make P5 shallow Conv (paper-friendly)
    # -------------------------
    targets_head = {"p3": None, "p4": None, "p5": None}

    for i, m in enumerate(layers):
        meta = get_c3k2_meta(m)
        if meta is None:
            continue
        c1, c2 = meta

        # focus head/neck area (after backbone)
        if i <= 10:
            continue

        # match your known head blocks
        if c1 == 256 and c2 == 64 and targets_head["p3"] is None:
            targets_head["p3"] = i
        elif c1 == 192 and c2 == 128 and targets_head["p4"] is None:
            targets_head["p4"] = i
        elif c1 == 384 and c2 == 256 and targets_head["p5"] is None:
            targets_head["p5"] = i

    print("[INFO] head targets:", targets_head)

    if targets_head["p3"] is None or targets_head["p4"] is None or targets_head["p5"] is None:
        raise RuntimeError(f"Không tìm đủ head blocks P3/P4/P5. Found={targets_head}")

    swap_mapboost_idxs = []

    def swap_to_mapboost(idx: int):
        old = layers[idx]
        c1, c2 = get_c3k2_meta(old)
        new = MAPBoost(c1, c2, s=1)
        for k in ("f", "i", "type"):
            if hasattr(old, k):
                setattr(new, k, getattr(old, k))
        new.training = old.training
        layers[idx] = new
        swap_mapboost_idxs.append(idx)
        print(f"[SWAP OK] HEAD idx={idx}: C3k2({c1}->{c2}) -> MAPBoost(full)")

    # P3, P4 = full MAPBoost
    swap_to_mapboost(targets_head["p3"])
    swap_to_mapboost(targets_head["p4"])

    # P5 = shallow Conv (light)
    p5_idx = targets_head["p5"]
    old_p5 = layers[p5_idx]
    new_p5 = Conv(384, 256, k=1, s=1)  # shallow 1x1 conv

    for k in ("f", "i", "type"):
        if hasattr(old_p5, k):
            setattr(new_p5, k, getattr(old_p5, k))
    new_p5.training = old_p5.training
    layers[p5_idx] = new_p5
    print(f"[SWAP OK] P5 idx={p5_idx}: C3k2(384->256) -> shallow Conv(1x1, 384->256)")

    # -------------------------
    # C) Swap WeightedConcat in place of Concat (keep Detect 3 heads)
    # -------------------------
    concat_idxs = [i for i, m in enumerate(layers) if m.__class__.__name__ == "Concat"]
    print("[INFO] Found Concat idxs:", concat_idxs)

    swap_wconcat_idxs = []

    def swap_to_wconcat(idx: int):
        old = layers[idx]
        dim = getattr(old, "d", 1)
        f = getattr(old, "f", None)
        n_inputs = len(f) if isinstance(f, (list, tuple)) else 2

        new = WeightedConcat(dimension=dim, n_inputs=n_inputs)

        for k in ("f", "i", "type"):
            if hasattr(old, k):
                setattr(new, k, getattr(old, k))
        new.training = old.training
        layers[idx] = new
        swap_wconcat_idxs.append(idx)
        print(f"[SWAP OK] Concat -> WeightedConcat at idx={idx}  (n_inputs={n_inputs}, dim={dim})")

    for idx in concat_idxs:
        swap_to_wconcat(idx)

    # 4) move to device
    model.model.to(device)  # type: ignore

    # -------------------------
    # Print model info
    # -------------------------
    print("\n================ MODEL INFO (after swap) ================\n", flush=True)
    model.model.info()

    print("\n================ LAYERS (after swap) ================\n", flush=True)

    swapped_idxs = {idx_p2, idx_p3, *swap_mapboost_idxs, p5_idx, *swap_wconcat_idxs}

    for i, m in enumerate(model.model.model):
        name = m.__class__.__name__
        f = getattr(m, "f", None)
        np_ = sum(p.numel() for p in m.parameters()) if isinstance(m, nn.Module) else 0

        c1 = c2 = None
        if hasattr(m, "cv1") and hasattr(m, "cv2") and hasattr(m.cv1, "conv") and hasattr(m.cv2, "conv"):
            try:
                c1 = m.cv1.conv.in_channels
                c2 = m.cv2.conv.out_channels
            except Exception:
                pass

        mark = "  <== SWAPPED" if i in swapped_idxs else ""

        if c1 is not None and c2 is not None:
            print(f"{i:3d}  f={f!s:>6}  {name:<20}  c1->c2: {c1:4d}->{c2:<4d}  params={np_:>8d}{mark}")
        else:
            print(f"{i:3d}  f={f!s:>6}  {name:<20}  params={np_:>8d}{mark}")

    print("[DEBUG] model param device:", next(model.model.parameters()).device)

    # -------------------------
    # Optimizer + Data
    # -------------------------
    optimizer = Adam(model.model.parameters(), lr=cfg.lr)

    train_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "train/images"),
        labels_dir=os.path.join(cfg.base_path, "train/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )
    val_dataset = AodDataset(
        images_dir=os.path.join(cfg.base_path, "valid/images"),
        labels_dir=os.path.join(cfg.base_path, "valid/labels"),
        images_ext=cfg.img_ext,
        labels_ext=cfg.label_ext,
        verbose_bad_labels=False,
    )

    pin = torch.cuda.is_available()
    workers = cfg.num_workers

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=workers,
        pin_memory=pin,
        persistent_workers=(workers > 0),
        drop_last=False,
    )

    # -------------------------
    # Train
    # -------------------------
    train_all = []

    best_score = float("inf")
    best_epoch = -1
    best_path = os.path.join(cfg.checkpoint_dir, "best.pt")
    last_path = os.path.join(cfg.checkpoint_dir, "last.pt")

    for epoch in range(1, cfg.epochs + 1):
        print(f"\n==================== EPOCH {epoch}/{cfg.epochs} ====================")

        train_df = train_one_epoch(
            model,
            optimizer,
            train_loader,
            device,
            epoch,
            max_train_steps=cfg.max_train_steps,
            debug_print_first_batch=(epoch == 1),
        )
        train_all.append(train_df)

        if len(train_df) > 0:
            train_score = train_df[["box", "cls", "dfl"]].mean().sum()
            print(f"[EPOCH {epoch}] train_score={train_score:.6f} | best_score={best_score:.6f}")

            if train_score < best_score:
                best_score = train_score
                best_epoch = epoch
                model.save(best_path)
                print(f"[BEST] updated → epoch {best_epoch}, saved to {best_path}")
        else:
            print("[TRAIN] no valid batches produced loss (all empty labels?)")

        model.save(last_path)
        ckpt_path = os.path.join(cfg.checkpoint_dir, f"epoch_{epoch}.pt")
        model.save(ckpt_path)
        print(f"[INFO] saved checkpoint: {ckpt_path} (and last.pt)")

    train_csv = os.path.join(cfg.checkpoint_dir, "train_losses.csv")
    pd.concat(train_all, axis=0, ignore_index=True).to_csv(train_csv, index=False)
    print(f"[INFO] saved train logs: {train_csv}")

    print("\n==================== TRAIN SUMMARY ====================")
    print(f"[BEST MODEL] epoch = {best_epoch}")
    print(f"[BEST MODEL] score = {best_score:.6f}")
    print(f"[BEST MODEL] path  = {best_path}")
    print(f"[LAST MODEL] path  = {last_path}")

    # -------------------------
    # Final evaluation
    # -------------------------
    print("\n==================== FINAL EVALUATION ====================")

    eval_ckpt = best_path if os.path.exists(best_path) else last_path
    print(f"[EVAL] loading checkpoint: {eval_ckpt}")

    eval_model = YOLO(eval_ckpt)
    eval_model.model.args = SimpleNamespace(box=15.0, cls=0.5, dfl=2.25)  # type: ignore
    eval_model.model.to(device)  # type: ignore

    val_loss_df = valid_loss_only(
        eval_model,
        val_loader,
        device,
        epoch=best_epoch if os.path.exists(best_path) else cfg.epochs,
        max_val_steps=cfg.max_val_steps,
    )
    val_csv = os.path.join(cfg.checkpoint_dir, "val_losses.csv")
    val_loss_df.to_csv(val_csv, index=False)
    if len(val_loss_df) > 0:
        print("[FINAL VALID-LOSS] mean:", val_loss_df[["box", "cls", "dfl"]].mean().to_dict())
    else:
        print("[FINAL VALID-LOSS] no valid batches (empty labels?)")
    print(f"[INFO] saved val losses: {val_csv}")

    try:
        print("[FINAL VALID-METRICS] running ultralytics val() ...")
        device_id = 0 if device.type == "cuda" else "cpu"
        metrics = yolo_val_metrics(eval_model, cfg.data_yaml, device_id=device_id, workers=cfg.num_workers)

        print(f"Precision: {metrics.box.mp:.4f}")
        print(f"Recall:    {metrics.box.mr:.4f}")
        print(f"mAP@0.5:   {metrics.box.map50:.4f}")
        print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
    except Exception as e:
        print(f"[WARN] final model.val() failed: {e}")

    return model


if __name__ == "__main__":

    def running_in_notebook():
        try:
            from IPython import get_ipython

            return get_ipython() is not None
        except Exception:
            return False

    if running_in_notebook():
        print("[INFO] Running in Kaggle/Colab Notebook → using hardcoded config")
        cfg = Config(
            base_path="/kaggle/input/aod-4-yolo",
            data_yaml="/kaggle/working/data.yaml",
            epochs=4,
            batch_size=4,
            lr=3e-4,
            num_workers=2,
            max_train_steps=1000,
            max_val_steps=0,
        )
        main(cfg)
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument("--base_path", type=str, required=True)
        parser.add_argument("--data_yaml", type=str, required=True)
        parser.add_argument("--epochs", type=int, default=2)
        parser.add_argument("--batch_size", type=int, default=4)
        parser.add_argument("--lr", type=float, default=3e-4)
        parser.add_argument("--num_workers", type=int, default=2)
        parser.add_argument("--img_ext", type=str, default="jpg")
        parser.add_argument("--label_ext", type=str, default="txt")
        parser.add_argument("--checkpoint_dir", type=str, default="checkpoints")
        parser.add_argument("--max_train_steps", type=int, default=0)
        parser.add_argument("--max_val_steps", type=int, default=0)

        args = parser.parse_args()
        cfg = Config(**vars(args))
        main(cfg)

[INFO] Running in Kaggle/Colab Notebook → using hardcoded config
[INFO] device = cuda:0
[INFO] CUDA available: True
[INFO] CUDA device: Tesla P100-PCIE-16GB

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultral

train epoch 1:   0%|          | 0/3941 [00:00<?, ?it/s]

[DEBUG] images device: cuda:0
[DEBUG] targets device: cuda:0
[DEBUG] model device: cuda:0
[EPOCH 1] train_score=14.816110 | best_score=inf
[BEST] updated → epoch 1, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_1.pt (and last.pt)

==================== EPOCH 2/4 ====================


train epoch 2:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 2] train_score=10.780889 | best_score=14.816110
[BEST] updated → epoch 2, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_2.pt (and last.pt)

==================== EPOCH 3/4 ====================


...

[EPOCH 3] train_score=9.705597 | best_score=10.780889
[BEST] updated → epoch 3, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_3.pt (and last.pt)

==================== EPOCH 4/4 ====================


train epoch 4:   0%|          | 0/3941 [00:00<?, ?it/s]

[EPOCH 4] train_score=9.101999 | best_score=9.705597
[BEST] updated → epoch 4, saved to checkpoints/best.pt
[INFO] saved checkpoint: checkpoints/epoch_4.pt (and last.pt)
[INFO] saved train logs: checkpoints/train_losses.csv

==================== TRAIN SUMMARY ====================
[BEST MODEL] epoch = 4
[BEST MODEL] score = 9.101999
[BEST MODEL] path  = checkpoints/best.pt
[LAST MODEL] path  = checkpoints/last.pt

==================== FINAL EVALUATION ====================
[EVAL] loading checkpoint: checkpoints/best.pt


valid(loss) epoch 4:   0%|          | 0/1129 [00:00<?, ?it/s]

[FINAL VALID-LOSS] mean: {'box': 3.9491081273074644, 'cls': 2.1436990570059806, 'dfl': 3.359693179002257}
[INFO] saved val losses: checkpoints/val_losses.csv
[FINAL VALID-METRICS] running ultralytics val() ...
Ultralytics 8.4.9 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
custom_YOLOv11n summary: 119 layers, 2,608,980 parameters, 879,644 gradients, 8.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 38.5±11.2 MB/s, size: 39.8 KB)
val: Scanning /kaggle/input/aod-4-yolo/valid/labels... 4514 images, 125 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4514/4514 822.3it/s 5.5s<0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/aod-4-yolo/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 283/283 10.9it/s 25.9s0.2ss
                   all       4514       6369      0.344      0.319      0.256     0.0964
Speed: 0.7ms preprocess, 2.6ms inference, 0.0ms loss, 0.8ms p